# Atividade 4 — Estudos de Caso em Deep Learning

**Parte 1 (Atividade 4.1):** detecção de COVID-19 em radiografias de tórax com transfer learning e avaliação de imagens sintéticas geradas por uma GAN condicional.
**Parte 2 (Atividade 4.2):** análise escrita de um caso de transfer learning para análise de tráfego urbano.

Dataset: [COVID-19 Radiography Database (Kaggle)](https://www.kaggle.com/datasets/tawsifurrahman/covid19-radiography-database).

## 1. Introdução e objetivos

### Cenário proposto (dados do enunciado, **não** resultados deste notebook)

| Item | Cenário original |
|---|---|
| Amostra | 1.200 radiografias (840 Normal, 240 Viral Pneumonia, 120 COVID-19) |
| Divisão | 80% treino / 20% validação, **não estratificada** |
| Modelo | ResNet-18 treinada do zero |
| Otimização | SGD, learning rate fixo de 0,01, 15 épocas |
| Data augmentation | Nenhuma |
| Avaliação | Somente acurácia global |
| Resultados relatados | 93% de acurácia no treino, 61% na validação |
| Queixa clínica | O modelo parece ignorar casos positivos de COVID-19 |

> **Importante:** os números da tabela acima pertencem ao cenário do enunciado. Os resultados efetivamente obtidos neste notebook são calculados nas células de código e registrados separadamente (arquivos JSON/CSV em `RESULTS_DIR`). Nenhum valor experimental é escrito manualmente no texto.

### Objetivos

1. Reproduzir um experimento **controlado** com a mesma composição de classes do cenário (840/240/120).
2. Diagnosticar as possíveis causas do baixo desempenho na classe COVID-19.
3. Treinar uma **GAN condicional (cGAN)** apenas com imagens reais de treinamento e gerar radiografias sintéticas da classe COVID-19.
4. Comparar dois experimentos que diferem **somente** na inclusão das imagens sintéticas:
   - **Experimento A:** apenas imagens reais de treinamento.
   - **Experimento B:** imagens reais de treinamento + imagens sintéticas de COVID-19.
5. Usar o **recall da classe COVID-19** como métrica primária, sem ignorar precisão, F1, macro F1 e matriz de confusão.
6. Propor um plano de melhorias e analisar o caso de tráfego urbano (Atividade 4.2).

### Roteiro do notebook

1. Introdução e objetivos · 2. Ambiente e reprodutibilidade · 3. Download e inspeção do dataset · 4. Seleção das 1.200 imagens · 5. Divisão dos dados e análise de vazamentos · 6. Diagnóstico dos problemas técnicos · 7. Preparação dos dados e métricas · 8. Treinamento do baseline · 9. Treinamento da GAN condicional · 10. Inspeção das imagens sintéticas · 11. Experimento A · 12. Experimento B · 13. Comparação e discussão · 14. Plano integrado de melhorias · 15. Atividade 4.2 · 16. Conclusão e limitações

> **Aviso:** este é um trabalho acadêmico. Nenhum modelo treinado aqui está apto a apoiar diagnóstico médico.

## 2. Ambiente e reprodutibilidade

No Colab, selecione **Ambiente de execução > Alterar o tipo de ambiente de execução > GPU T4**. O notebook também roda em CPU (fallback), porém de forma muito mais lenta.

A célula abaixo importa as bibliotecas e exibe as versões do Python, PyTorch, torchvision e CUDA. Somente bibliotecas comuns do ecossistema PyTorch/científico são usadas (`torch`, `torchvision`, `numpy`, `pandas`, `scikit-learn`, `matplotlib`, `seaborn`, `Pillow`). A CLI `kaggle` é instalada apenas se não estiver disponível.

In [ ]:
import os
import io
import sys
import json
import time
import copy
import shutil
import random
import hashlib
import platform
import subprocess
import zipfile
from pathlib import Path
from contextlib import contextmanager
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from IPython.display import display, Markdown

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision
from torchvision import transforms as T
from torchvision.models import resnet18, ResNet18_Weights
from torchvision.utils import make_grid, save_image

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support, f1_score,
                             confusion_matrix, classification_report)
from tqdm.auto import tqdm

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'Python      : {platform.python_version()}')
print(f'PyTorch     : {torch.__version__}')
print(f'torchvision : {torchvision.__version__}')
print(f'CUDA (build): {torch.version.cuda}')
print(f'cuDNN       : {torch.backends.cudnn.version() if torch.cuda.is_available() else "n/d"}')
print(f'Dispositivo : {DEVICE}')
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'GPU         : {props.name} ({props.total_memory / 1024**3:.1f} GB)')
else:
    print('GPU não disponível: o notebook executará em CPU (bem mais lento).')

### Parâmetros principais

Todos os parâmetros que controlam o experimento ficam concentrados na célula abaixo:

- **`SEED`**: semente única para Python, NumPy e PyTorch (seleção da amostra, divisões, inicialização, embaralhamento e geração de ruído da GAN).
- **`N_PER_CLASS`**: tamanho da amostra experimental por classe (840/240/120, como no cenário).
- **`VAL_FRACTION` / `INTERNAL_VAL_FRACTION`**: 20% da amostra para validação real; ~15% do treino separado como **validação interna** (usada apenas para escolher o checkpoint).
- **`DHASH_MAX_DISTANCE`**: limiar (em bits, de 64) do hash perceptual usado para agrupar imagens quase idênticas antes da divisão.
- **Classificador** (`CLS_*`): resolução 224×224 (nativa da ResNet-18 pré-treinada), lote 32, 15 épocas (mesmo número do cenário), Adam com lr 1e-3 e decaimento cosseno.
- **cGAN** (`GAN_*`): resolução 64×64, vetor latente de 100 dimensões, lote 64, Adam (lr 2e-4, β₁=0,5).
- **`N_SYNTHETIC`**: quantidade de imagens sintéticas de COVID-19 adicionadas no Experimento B (ponto de partida: 120; **não** é um valor garantidamente ideal).
- **`QUICK_RUN`**: reduz drasticamente épocas para um teste rápido do fluxo (não usar para conclusões).

Para salvar os resultados no Google Drive, monte o Drive e aponte `BASE_DIR` para uma pasta dentro de `/content/drive/MyDrive`.

In [ ]:
# ----------------------------- Reprodutibilidade -----------------------------
SEED = 42

# ----------------------------- Diretórios ------------------------------------
IN_COLAB = 'google.colab' in sys.modules
BASE_DIR = Path(os.environ.get('ATV4_BASE_DIR', '/content' if IN_COLAB else './atv4_workspace')).resolve()
DATA_DIR = BASE_DIR / 'data' / 'covid19-radiography'     # dataset extraído
DOWNLOAD_DIR = BASE_DIR / 'data' / 'downloads'            # arquivo .zip do Kaggle
RESULTS_DIR = BASE_DIR / 'results_atividade4'             # tudo que o notebook produz
CHECKPOINT_DIR = RESULTS_DIR / 'checkpoints'
FIGURES_DIR = RESULTS_DIR / 'figures'
GAN_SAMPLES_DIR = RESULTS_DIR / 'gan_samples'
SYNTHETIC_DIR = RESULTS_DIR / 'synthetic_covid'
for directory in [DATA_DIR, DOWNLOAD_DIR, RESULTS_DIR, CHECKPOINT_DIR, FIGURES_DIR, GAN_SAMPLES_DIR, SYNTHETIC_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

KAGGLE_DATASET_SLUG = 'tawsifurrahman/covid19-radiography-database'

# ----------------------------- Classes e amostra -----------------------------
CLASS_NAMES = ['Normal', 'Viral Pneumonia', 'COVID']      # ordem fixa dos rótulos
CLASS_TO_IDX = {name: idx for idx, name in enumerate(CLASS_NAMES)}
COVID_IDX = CLASS_TO_IDX['COVID']
N_PER_CLASS = {'Normal': 840, 'Viral Pneumonia': 240, 'COVID': 120}

# ----------------------------- Divisão dos dados -----------------------------
VAL_FRACTION = 0.20             # validação real (nunca usada no treino nem na GAN)
INTERNAL_VAL_FRACTION = 0.15    # fração do treino para seleção de checkpoint
DHASH_MAX_DISTANCE = 4          # distância de Hamming máxima para considerar imagens "quase idênticas"

# ----------------------------- Classificador ---------------------------------
CLS_IMAGE_SIZE = 224
CLS_BATCH_SIZE = 32
CLS_EPOCHS = 15
CLS_LR = 1e-3
CLS_WEIGHT_DECAY = 1e-4
CHECKPOINT_METRIC = 'internal_val_macro_f1'   # definida ANTES do treinamento
NUM_WORKERS = 2
# As imagens ficam em um cache global; os workers precisam herdá-lo via 'fork' (padrão no Linux/Colab)
import multiprocessing as mp
LOADER_MP_CONTEXT = 'fork' if 'fork' in mp.get_all_start_methods() else None
if LOADER_MP_CONTEXT is None:
    NUM_WORKERS = 0

# ----------------------------- GAN condicional -------------------------------
GAN_IMAGE_SIZE = 64
GAN_Z_DIM = 100
GAN_EMB_DIM = 50
GAN_BATCH_SIZE = 64
GAN_EPOCHS = 400
GAN_LR = 2e-4
GAN_BETAS = (0.5, 0.999)
GAN_REAL_LABEL = 0.9            # label smoothing unilateral para estabilizar o discriminador
GAN_N_PREVIEWS = 5              # quantas vezes exibir amostras durante o treino
N_SYNTHETIC = 120               # quantidade de imagens sintéticas de COVID-19 para o Experimento B

# ----------------------------- Execução rápida (apenas teste do fluxo) -------
QUICK_RUN = os.environ.get('ATV4_QUICK_RUN', '0') == '1'
if QUICK_RUN:
    CLS_EPOCHS, GAN_EPOCHS = 2, 4
    print('ATENÇÃO: QUICK_RUN ativo — resultados servem apenas para testar o fluxo.')

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
IMAGE_EXTENSIONS = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'}

print(f'BASE_DIR    : {BASE_DIR}')
print(f'DATA_DIR    : {DATA_DIR}')
print(f'RESULTS_DIR : {RESULTS_DIR}')
print(f'N_SYNTHETIC : {N_SYNTHETIC} | CLS_EPOCHS: {CLS_EPOCHS} | GAN_EPOCHS: {GAN_EPOCHS}')

In [ ]:
def set_seed(seed: int = SEED) -> None:
    """Fixa as sementes de Python, NumPy e PyTorch (CPU e GPU)."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    # Desativa heurísticas não determinísticas do cuDNN (custo pequeno de velocidade)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def seed_worker(worker_id: int) -> None:
    """Semente de cada worker do DataLoader derivada da semente do PyTorch."""
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


EXEC_TIMES = {}


@contextmanager
def timer(name: str):
    """Mede e registra o tempo de execução de uma etapa."""
    start = time.perf_counter()
    yield
    elapsed = time.perf_counter() - start
    EXEC_TIMES[name] = elapsed
    print(f'⏱  {name}: {elapsed / 60:.1f} min ({elapsed:.0f} s)')


def save_fig(name: str) -> None:
    """Salva a figura atual em FIGURES_DIR."""
    plt.savefig(FIGURES_DIR / f'{name}.png', dpi=120, bbox_inches='tight')


def save_json(obj, path: Path) -> None:
    with open(path, 'w', encoding='utf-8') as file:
        json.dump(obj, file, indent=2, ensure_ascii=False)


set_seed(SEED)
sns.set_theme(style='whitegrid')
print(f'Semente global definida: {SEED}')

## 3. Download e inspeção do dataset

### Como obter e configurar o token do Kaggle (sem expor credenciais)

1. Acesse [kaggle.com](https://www.kaggle.com) → clique na sua foto → **Settings**.
2. Na seção **API**, clique em **Create New Token**. Dependendo da versão do site:
   - é baixado um arquivo `kaggle.json` com os campos `username` e `key`; **ou**
   - é exibido um token de API (formato novo).
3. No Colab, abra o painel **🔑 Secrets** (ícone de chave na barra lateral esquerda) e crie:
   - `KAGGLE_USERNAME` = valor de `username` e `KAGGLE_KEY` = valor de `key`; **ou**
   - `KAGGLE_API_TOKEN` = token no formato novo.
4. Ative a opção **Notebook access** para esses secrets.
5. Aceite os termos do dataset na página do Kaggle, se solicitado.

As credenciais são lidas em tempo de execução via `google.colab.userdata` e colocadas apenas em variáveis de ambiente do processo. **Nenhuma credencial é escrita no código ou salva no notebook.** Fora do Colab, o arquivo `~/.kaggle/kaggle.json` ou as mesmas variáveis de ambiente também funcionam.

Se o dataset já estiver extraído em `DATA_DIR`, o download é ignorado.

In [ ]:
def configure_kaggle_credentials() -> bool:
    """Carrega credenciais do Kaggle dos Secrets do Colab para variáveis de ambiente."""
    try:
        from google.colab import userdata  # disponível apenas no Colab
        for key in ['KAGGLE_USERNAME', 'KAGGLE_KEY', 'KAGGLE_API_TOKEN']:
            try:
                value = userdata.get(key)
            except Exception:
                value = None
            if value:
                os.environ[key] = value
    except ImportError:
        pass
    has_pair = bool(os.environ.get('KAGGLE_USERNAME') and os.environ.get('KAGGLE_KEY'))
    has_token = bool(os.environ.get('KAGGLE_API_TOKEN'))
    has_file = (Path.home() / '.kaggle' / 'kaggle.json').exists()
    return has_pair or has_token or has_file


def count_images(root: Path) -> int:
    return sum(1 for p in root.rglob('*') if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS)


def download_dataset() -> None:
    """Baixa e extrai o dataset via API oficial do Kaggle, se ainda não estiver presente."""
    if count_images(DATA_DIR) > 0:
        print(f'Dataset já presente em {DATA_DIR} — download ignorado.')
        return
    if not configure_kaggle_credentials():
        raise RuntimeError(
            'Credenciais do Kaggle não encontradas. Configure KAGGLE_USERNAME/KAGGLE_KEY ou '
            'KAGGLE_API_TOKEN nos Secrets do Colab (ver instruções acima) e execute novamente.')
    if shutil.which('kaggle') is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'kaggle'], check=True)
    print(f'Baixando {KAGGLE_DATASET_SLUG} ...')
    subprocess.run(['kaggle', 'datasets', 'download', '-d', KAGGLE_DATASET_SLUG,
                    '-p', str(DOWNLOAD_DIR)], check=True)
    zips = sorted(DOWNLOAD_DIR.glob('*.zip'))
    if not zips:
        raise RuntimeError(f'Nenhum .zip encontrado em {DOWNLOAD_DIR} após o download.')
    print(f'Extraindo {zips[0].name} ...')
    with zipfile.ZipFile(zips[0]) as archive:
        archive.extractall(DATA_DIR)
    print('Extração concluída.')


with timer('download_dataset'):
    download_dataset()

### Inspeção da estrutura real do dataset

Antes de assumir caminhos fixos, listamos os diretórios que contêm imagens e os arquivos de metadados. Versões recentes do dataset costumam ter a forma `COVID-19_Radiography_Dataset/<Classe>/images` e `<Classe>/masks` (máscaras de segmentação pulmonar, que **não** são radiografias e devem ser descartadas), além de planilhas `<Classe>.metadata.xlsx`. O código abaixo não depende dessa forma exata: ele procura os diretórios das classes pelo nome.

In [ ]:
def summarize_structure(root: Path, max_rows: int = 30) -> pd.DataFrame:
    """Conta imagens por diretório (relativo à raiz) para revelar a organização real."""
    counts = {}
    for path in root.rglob('*'):
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS:
            rel = str(path.parent.relative_to(root))
            counts[rel] = counts.get(rel, 0) + 1
    table = (pd.DataFrame(sorted(counts.items()), columns=['diretório', 'n_imagens'])
             .sort_values('n_imagens', ascending=False).reset_index(drop=True))
    return table.head(max_rows)


structure_table = summarize_structure(DATA_DIR)
display(structure_table)
metadata_files = sorted(DATA_DIR.rglob('*.xlsx')) + sorted(DATA_DIR.rglob('*.csv'))
print('Arquivos de metadados encontrados:')
for path in metadata_files:
    print('  -', path.relative_to(DATA_DIR))

In [ ]:
# Apelidos aceitos para cada classe (comparação sem maiúsculas, '_' e '-')
CLASS_ALIASES = {
    'Normal': {'normal'},
    'Viral Pneumonia': {'viral pneumonia', 'viralpneumonia'},
    'COVID': {'covid', 'covid 19', 'covid19'},
}
EXCLUDED_DIR_NAMES = {'masks', 'mask', 'lung_masks'}


def normalize_name(name: str) -> str:
    return name.lower().replace('_', ' ').replace('-', ' ').strip()


def find_class_image_dirs(root: Path) -> dict:
    """Localiza automaticamente o diretório de imagens de cada classe.

    Para cada classe, considera diretórios cujo nome corresponde a um apelido; se houver
    subpasta `images`, ela é usada (descartando `masks`). Havendo mais de um candidato,
    escolhe o com mais imagens e avisa.
    """
    found = {}
    for class_name, aliases in CLASS_ALIASES.items():
        candidates = []
        for directory in root.rglob('*'):
            if directory.is_dir() and normalize_name(directory.name) in aliases:
                image_dir = directory / 'images' if (directory / 'images').is_dir() else directory
                n_images = sum(1 for p in image_dir.iterdir()
                               if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS)
                if n_images > 0:
                    candidates.append((n_images, image_dir))
        if not candidates:
            raise FileNotFoundError(
                f"Diretório da classe '{class_name}' não encontrado em {root}.\n"
                f"Verifique se o dataset foi extraído corretamente e se existem pastas como "
                f"'COVID/images', 'Normal/images' e 'Viral Pneumonia/images'. "
                f"Estrutura encontrada: veja a tabela da célula anterior.")
        candidates.sort(key=lambda item: (-item[0], str(item[1])))
        if len(candidates) > 1:
            print(f"Aviso: {len(candidates)} diretórios candidatos para '{class_name}'; usando {candidates[0][1]}")
        found[class_name] = candidates[0][1]
    return found


CLASS_DIRS = find_class_image_dirs(DATA_DIR)
for class_name, directory in CLASS_DIRS.items():
    print(f'{class_name:16s} -> {directory.relative_to(DATA_DIR)}')

### Inventário: extensões, dimensões, imagens corrompidas e duplicatas

Para cada imagem das três classes calculamos: extensão, dimensões, modo de cor, hash MD5 do arquivo (duplicatas exatas) e se o arquivo pode ser decodificado. A leitura usa threads para reduzir o tempo de I/O; nenhum pixel fica em memória nesta etapa.

In [ ]:
def inspect_image(args):
    """Lê um arquivo, calcula MD5 e verifica se a imagem é decodificável."""
    path, class_name = args
    record = {'path': str(path), 'file_name': path.name, 'class_name': class_name,
              'extension': path.suffix.lower(), 'width': np.nan, 'height': np.nan,
              'mode': None, 'md5': None, 'corrupted': False, 'error': None}
    try:
        data = path.read_bytes()
        record['md5'] = hashlib.md5(data).hexdigest()
        with Image.open(io.BytesIO(data)) as image:
            image.verify()                     # checa a integridade do arquivo
        with Image.open(io.BytesIO(data)) as image:
            image.load()                       # decodifica de fato os pixels
            record['width'], record['height'] = image.size
            record['mode'] = image.mode
    except Exception as error:                 # arquivo truncado, formato inválido etc.
        record['corrupted'] = True
        record['error'] = f'{type(error).__name__}: {error}'
    return record


tasks = [(path, class_name) for class_name, directory in CLASS_DIRS.items()
         for path in sorted(directory.iterdir())
         if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS]

with timer('inventario_dataset'):
    with ThreadPoolExecutor(max_workers=8) as executor:
        inventory = pd.DataFrame(list(tqdm(executor.map(inspect_image, tasks),
                                           total=len(tasks), desc='Inspecionando imagens')))

print(f'Total de imagens nas três classes: {len(inventory)}')
print('\nExtensões por classe:')
display(pd.crosstab(inventory['class_name'], inventory['extension']))
print('Modo de cor por classe:')
display(pd.crosstab(inventory['class_name'], inventory['mode'].fillna('n/d')))
print('Dimensões mais frequentes (largura × altura):')
display(inventory.assign(dim=inventory['width'].astype('Int64').astype(str) + '×' +
                         inventory['height'].astype('Int64').astype(str))
        .groupby(['class_name', 'dim']).size().rename('n').reset_index()
        .sort_values('n', ascending=False).head(10))

corrupted = inventory[inventory['corrupted']]
print(f'Imagens corrompidas: {len(corrupted)}')
if len(corrupted):
    display(corrupted[['path', 'class_name', 'error']].head(20))

valid_inventory = inventory[~inventory['corrupted']]
dup_counts = valid_inventory.groupby('md5')['path'].transform('count')
dup_classes = valid_inventory.groupby('md5')['class_name'].transform('nunique')
print(f'Imagens envolvidas em duplicatas exatas (mesmo MD5): {(dup_counts > 1).sum()}')
print(f'  ...das quais com o mesmo conteúdo em classes diferentes: {(dup_classes > 1).sum()}')

### Metadados e fontes das imagens

As planilhas `*.metadata.xlsx` informam, para cada arquivo, o formato, o tamanho e a **URL de origem** (repositório/instituição de onde a imagem foi coletada). Elas **não** trazem identificador de paciente. A fonte é útil para dois diagnósticos:

- se cada classe vier predominantemente de fontes diferentes, o classificador pode aprender **atalhos** (tipo de equipamento, marcações, pós-processamento) em vez de achados clínicos;
- a fonte serve como agrupamento grosseiro, mas não substitui o ID de paciente.

In [ ]:
def load_metadata(files) -> pd.DataFrame:
    """Lê as planilhas de metadados (se existirem) e padroniza nome do arquivo e fonte."""
    frames = []
    for path in files:
        if path.suffix.lower() != '.xlsx':
            continue
        try:
            frame = pd.read_excel(path)
        except Exception as error:
            print(f'Não foi possível ler {path.name}: {error}')
            continue
        columns = {c: str(c).strip().lower() for c in frame.columns}
        name_col = next((c for c, n in columns.items() if 'file' in n and 'name' in n), None)
        url_col = next((c for c, n in columns.items() if 'url' in n), None)
        if name_col is None:
            print(f'Planilha {path.name} sem coluna de nome de arquivo — ignorada.')
            continue
        frames.append(pd.DataFrame({
            'stem_key': frame[name_col].astype(str).str.strip().str.lower(),
            'source_url': frame[url_col].astype(str) if url_col else 'desconhecida',
        }))
    if not frames:
        return pd.DataFrame(columns=['stem_key', 'source_url'])
    return pd.concat(frames).drop_duplicates('stem_key')


def url_to_source(url: str) -> str:
    """Reduz a URL ao domínio/repositório para agrupar fontes."""
    if not isinstance(url, str) or not url.startswith('http'):
        return 'desconhecida'
    parts = url.split('/')
    domain = parts[2].replace('www.', '')
    if 'github.com' in domain and len(parts) > 4:
        return f'github/{parts[3]}/{parts[4]}'
    return domain


metadata = load_metadata(metadata_files)
inventory['stem_key'] = inventory['file_name'].str.rsplit('.', n=1).str[0].str.lower()
inventory = inventory.merge(metadata, on='stem_key', how='left')
inventory['source'] = inventory['source_url'].map(url_to_source)
print(f'Imagens com metadados de fonte: {(inventory["source"] != "desconhecida").sum()} de {len(inventory)}')
patient_cols = [c for c in metadata.columns if 'patient' in c.lower()]
print(f'Colunas com identificador de paciente nos metadados: {patient_cols or "nenhuma"}')
display(pd.crosstab(inventory['source'], inventory['class_name']))

In [ ]:
original_dist = (inventory[~inventory['corrupted']]['class_name'].value_counts()
                 .reindex(CLASS_NAMES).rename('n_original').to_frame())
original_dist['%_original'] = (100 * original_dist['n_original'] / original_dist['n_original'].sum()).round(2)
print('Distribuição ORIGINAL do dataset (três classes usadas, imagens válidas):')
display(original_dist)


def show_class_examples(frame: pd.DataFrame, n_per_class: int = 4, title: str = '', name: str = None):
    """Exibe n_per_class exemplos de cada classe em uma grade."""
    fig, axes = plt.subplots(len(CLASS_NAMES), n_per_class, figsize=(2.6 * n_per_class, 2.8 * len(CLASS_NAMES)))
    for row, class_name in enumerate(CLASS_NAMES):
        subset = frame[frame['class_name'] == class_name]
        examples = subset.sample(n=min(n_per_class, len(subset)), random_state=SEED)
        for col in range(n_per_class):
            ax = axes[row, col]
            ax.axis('off')
            if col < len(examples):
                entry = examples.iloc[col]
                with Image.open(entry['path']) as image:
                    ax.imshow(image.convert('L'), cmap='gray')
                ax.set_title(f"{class_name}\n{entry['file_name']}", fontsize=8)
    fig.suptitle(title, fontsize=13)
    plt.tight_layout()
    if name:
        save_fig(name)
    plt.show()


show_class_examples(inventory[~inventory['corrupted']], title='Exemplos reais de cada classe', name='exemplos_classes')

## 4. Seleção das 1.200 imagens

Regras da seleção:

1. Descartar imagens corrompidas.
2. Duplicatas exatas (mesmo MD5) **dentro** da mesma classe: manter apenas uma cópia, para que a mesma imagem não caia no treino e na validação.
3. Conteúdo idêntico em **classes diferentes**: descartar todas as cópias (rótulo ambíguo).
4. Amostrar, com semente fixa, exatamente 840 Normal, 240 Viral Pneumonia e 120 COVID.
5. Se alguma classe não tiver imagens suficientes, interromper com erro — **nunca** duplicar imagens silenciosamente.

In [ ]:
def build_candidate_pool(frame: pd.DataFrame) -> pd.DataFrame:
    pool = frame[~frame['corrupted']].copy()
    ambiguous_md5 = pool.groupby('md5')['class_name'].nunique()
    ambiguous_md5 = set(ambiguous_md5[ambiguous_md5 > 1].index)
    removed_ambiguous = pool['md5'].isin(ambiguous_md5).sum()
    pool = pool[~pool['md5'].isin(ambiguous_md5)]
    before = len(pool)
    pool = pool.sort_values('path').drop_duplicates('md5', keep='first')
    print(f'Removidas por conteúdo idêntico em classes diferentes: {removed_ambiguous}')
    print(f'Removidas por duplicata exata na mesma classe: {before - len(pool)}')
    return pool.reset_index(drop=True)


def select_sample(pool: pd.DataFrame, n_per_class: dict, seed: int) -> pd.DataFrame:
    available = pool['class_name'].value_counts()
    missing = {c: (int(available.get(c, 0)), n) for c, n in n_per_class.items() if available.get(c, 0) < n}
    if missing:
        details = '; '.join(f'{c}: disponível {a}, necessário {n}' for c, (a, n) in missing.items())
        raise RuntimeError(f'Imagens insuficientes para a amostra experimental ({details}). '
                           'A execução foi interrompida para não duplicar imagens.')
    parts = [pool[pool['class_name'] == c].sample(n=n, random_state=seed) for c, n in n_per_class.items()]
    sample = pd.concat(parts).sample(frac=1, random_state=seed).reset_index(drop=True)
    sample['label'] = sample['class_name'].map(CLASS_TO_IDX).astype(int)
    sample['is_synthetic'] = False
    return sample


candidate_pool = build_candidate_pool(inventory)
sample_df = select_sample(candidate_pool, N_PER_CLASS, SEED)

sample_dist = sample_df['class_name'].value_counts().reindex(CLASS_NAMES).rename('n_amostra').to_frame()
sample_dist['%_amostra'] = (100 * sample_dist['n_amostra'] / sample_dist['n_amostra'].sum()).round(2)
comparison_dist = original_dist.join(sample_dist)
comparison_dist.loc['Total'] = comparison_dist.sum(numeric_only=True)
print(f'Amostra experimental: {len(sample_df)} imagens')
display(comparison_dist)

fig, ax = plt.subplots(figsize=(8, 4))
comparison_dist.drop('Total')[['%_original', '%_amostra']].plot.bar(ax=ax, color=['#9aa5b1', '#2a6fdb'])
ax.set_title('Distribuição de classes: dataset original × amostra experimental')
ax.set_xlabel('Classe')
ax.set_ylabel('Proporção (%)')
ax.legend(['Dataset original (3 classes)', 'Amostra experimental (1.200)'])
plt.xticks(rotation=0)
plt.tight_layout()
save_fig('distribuicao_original_vs_amostra')
plt.show()

**Leitura:** a proporção da amostra (70% / 20% / 10%) foi imposta pelo cenário e **não** corresponde à distribuição original do dataset completo. Conclusões sobre prevalência ou valor preditivo não podem ser transferidas da amostra para o dataset original, nem para uma população clínica real.

## 5. Divisão dos dados e análise de possíveis vazamentos

### Agrupamento antes da divisão

Os metadados públicos **não** trazem identificador de paciente; portanto, uma divisão por paciente confiável **não é possível**. Essa limitação é registrada explicitamente. Como mitigação parcial:

1. As duplicatas exatas já foram removidas na seleção.
2. Calculamos um **hash perceptual (dHash, 64 bits)** de cada imagem e unimos em um mesmo grupo as imagens cuja distância de Hamming é ≤ `DHASH_MAX_DISTANCE`. Isso captura cópias redimensionadas, recomprimidas ou levemente recortadas — um sinal comum de imagens repetidas do mesmo exame.
3. A divisão usa **`StratifiedGroupKFold`**: preserva as proporções de classe e garante que um grupo inteiro fique em um único conjunto.

O hash perceptual **não** identifica radiografias diferentes do mesmo paciente feitas em dias distintos; o risco de vazamento entre pacientes permanece e é discutido na Seção 6.

In [ ]:
def dhash_bits(path: str, hash_size: int = 8) -> np.ndarray:
    """Hash por diferença: compara pixels vizinhos de uma miniatura 9×8 em tons de cinza."""
    with Image.open(path) as image:
        small = image.convert('L').resize((hash_size + 1, hash_size), Image.BILINEAR)
    pixels = np.asarray(small, dtype=np.int16)
    return (pixels[:, 1:] > pixels[:, :-1]).flatten()


def hamming_matrix(bits_a: np.ndarray, bits_b: np.ndarray) -> np.ndarray:
    """Distância de Hamming entre todos os pares (via produto matricial, sem laço duplo)."""
    a = bits_a.astype(np.float32)
    b = bits_b.astype(np.float32)
    agreements = a @ b.T + (1 - a) @ (1 - b).T
    return (a.shape[1] - agreements).astype(np.int32)


def union_find_groups(n: int, pairs) -> np.ndarray:
    parent = list(range(n))

    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i

    for i, j in pairs:
        root_i, root_j = find(i), find(j)
        if root_i != root_j:
            parent[root_j] = root_i
    return np.array([find(i) for i in range(n)])


with timer('hash_perceptual'):
    sample_bits = np.stack([dhash_bits(p) for p in tqdm(sample_df['path'], desc='dHash')])
distances = hamming_matrix(sample_bits, sample_bits)
np.fill_diagonal(distances, 999)
near_pairs = np.argwhere(np.triu(distances <= DHASH_MAX_DISTANCE, k=1))
sample_df['group_id'] = union_find_groups(len(sample_df), near_pairs)

group_sizes = sample_df['group_id'].value_counts()
print(f'Pares quase idênticos (Hamming ≤ {DHASH_MAX_DISTANCE}): {len(near_pairs)}')
print(f'Grupos com mais de uma imagem: {(group_sizes > 1).sum()} | maior grupo: {group_sizes.max()} imagens')
print(f'Total de grupos: {sample_df["group_id"].nunique()} para {len(sample_df)} imagens')
if group_sizes.max() > 0.05 * len(sample_df):
    print('Aviso: grupo muito grande — o limiar pode estar agrupando radiografias apenas semelhantes.')

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist(distances.min(axis=1), bins=range(0, 34), color='#2a6fdb')
ax.axvline(DHASH_MAX_DISTANCE + 0.5, color='crimson', linestyle='--', label=f'limiar = {DHASH_MAX_DISTANCE}')
ax.set_title('Distância de Hamming até a imagem mais parecida da amostra')
ax.set_xlabel('Distância de Hamming (bits de 64)')
ax.set_ylabel('Número de imagens')
ax.legend()
plt.tight_layout()
save_fig('dhash_vizinho_mais_proximo')
plt.show()

if len(near_pairs):
    n_show = min(4, len(near_pairs))
    fig, axes = plt.subplots(n_show, 2, figsize=(5, 2.5 * n_show), squeeze=False)
    for row, (i, j) in enumerate(near_pairs[:n_show]):
        for col, idx in enumerate((i, j)):
            with Image.open(sample_df.loc[idx, 'path']) as image:
                axes[row, col].imshow(image.convert('L'), cmap='gray')
            axes[row, col].set_title(f"{sample_df.loc[idx, 'class_name']} | {sample_df.loc[idx, 'file_name']}", fontsize=7)
            axes[row, col].axis('off')
    fig.suptitle('Exemplos de pares agrupados como quase idênticos', fontsize=11)
    plt.tight_layout()
    plt.show()

In [ ]:
def stratified_group_split(frame: pd.DataFrame, fraction: float, seed: int):
    """Separa ~fraction das linhas mantendo grupos inteiros e a proporção de classes."""
    n_splits = max(2, int(round(1 / fraction)))
    splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    keep_idx, holdout_idx = next(splitter.split(frame, frame['label'], groups=frame['group_id']))
    return frame.iloc[keep_idx].reset_index(drop=True), frame.iloc[holdout_idx].reset_index(drop=True)


train_df, val_df = stratified_group_split(sample_df, VAL_FRACTION, SEED)
train_fit_df, internal_val_df = stratified_group_split(train_df, INTERNAL_VAL_FRACTION, SEED)
train_df['split'] = 'treino'
val_df['split'] = 'validacao'
train_fit_df['split'] = 'treino_ajuste'
internal_val_df['split'] = 'validacao_interna'


def split_table(frames: dict) -> pd.DataFrame:
    table = {}
    for name, frame in frames.items():
        counts = frame['class_name'].value_counts().reindex(CLASS_NAMES, fill_value=0)
        table[(name, 'n')] = counts
        table[(name, '%')] = (100 * counts / counts.sum()).round(1)
    out = pd.DataFrame(table)
    out.loc['Total'] = [frame.shape[0] if c[1] == 'n' else 100.0 for c in out.columns
                        for frame in [frames[c[0]]]]
    return out


display(split_table({'Treino (80%)': train_df, 'Validação real (20%)': val_df,
                     'Treino de ajuste': train_fit_df, 'Validação interna': internal_val_df}))

In [ ]:
def check_overlap(a: pd.DataFrame, b: pd.DataFrame, name_a: str, name_b: str) -> dict:
    """Verifica sobreposição de arquivos, conteúdo (MD5), grupos e quase-duplicatas entre dois conjuntos."""
    bits_a = np.stack([dhash_bits(p) for p in a['path']])
    bits_b = np.stack([dhash_bits(p) for p in b['path']])
    cross = hamming_matrix(bits_a, bits_b)
    result = {
        'conjuntos': f'{name_a} × {name_b}',
        'arquivos em comum': len(set(a['path']) & set(b['path'])),
        'MD5 em comum': len(set(a['md5']) & set(b['md5'])),
        'grupos em comum': len(set(a['group_id']) & set(b['group_id'])),
        f'pares dHash ≤ {DHASH_MAX_DISTANCE}': int((cross <= DHASH_MAX_DISTANCE).sum()),
        'menor distância dHash': int(cross.min()),
    }
    return result


overlap_report = pd.DataFrame([
    check_overlap(train_df, val_df, 'treino', 'validação'),
    check_overlap(train_fit_df, internal_val_df, 'treino de ajuste', 'validação interna'),
    check_overlap(internal_val_df, val_df, 'validação interna', 'validação'),
])
display(overlap_report)
assert overlap_report['arquivos em comum'].sum() == 0, 'Vazamento: arquivos repetidos entre conjuntos!'
assert overlap_report['MD5 em comum'].sum() == 0, 'Vazamento: conteúdo idêntico entre conjuntos!'
assert overlap_report['grupos em comum'].sum() == 0, 'Vazamento: grupo dividido entre conjuntos!'
print('Verificações de sobreposição aprovadas (arquivos, MD5 e grupos).')

print('\nFontes (metadados) por classe e conjunto — indício de possíveis atalhos de domínio:')
train_val_df = pd.concat([train_df, val_df], ignore_index=True)
display(pd.crosstab([train_val_df['split'], train_val_df['class_name']], train_val_df['source']))

split_columns = ['path', 'file_name', 'class_name', 'label', 'md5', 'source', 'group_id', 'split']
pd.concat([train_fit_df, internal_val_df, val_df])[split_columns].to_csv(RESULTS_DIR / 'splits.csv', index=False)
print(f'Divisão salva em {RESULTS_DIR / "splits.csv"}')

**Regras de uso de cada conjunto (fixadas antes de qualquer treinamento):**

| Conjunto | Uso permitido | Uso proibido |
|---|---|---|
| Treino de ajuste | Atualizar pesos do classificador; treinar a cGAN | — |
| Validação interna | Selecionar o melhor checkpoint (macro F1) | Treinar classificador ou GAN |
| Validação real (20%) | **Somente** a avaliação final de A e B, uma única vez | Treinar a GAN, escolher imagens sintéticas, escolher época, ajustar hiperparâmetros |

Como não há conjunto de teste independente além da validação real, a validação real cumpre o papel de conjunto de avaliação; ela não é consultada durante o desenvolvimento dos modelos.

## 6. Diagnóstico dos problemas técnicos do cenário original

Legenda: **Comprovado** = decorre diretamente dos dados do cenário; **Risco** = plausível, mas precisa ser investigado.

| # | Problema | Status | Evidência no cenário | Consequência possível |
|---|---|---|---|---|
| 1 | **Desbalanceamento de classes** (70% / 20% / 10%) | Comprovado | 840 / 240 / 120 imagens | A perda é dominada pela classe Normal; prever "Normal" para tudo já gera 70% de acurácia. Muitos **falsos negativos** de COVID-19. |
| 2 | **Divisão não estratificada** | Comprovado | Enunciado | A validação pode ter muito poucos casos COVID (ou proporção diferente do treino). Com ~24 casos esperados, cada caso vale ~4 p.p. de recall; a métrica fica instável e enviesada. |
| 3 | **ResNet-18 treinada do zero** com 960 imagens | Comprovado | ~11 milhões de parâmetros para ~960 exemplos | Alta capacidade com pouca informação ⇒ memorização. Coerente com 93% no treino × 61% na validação (lacuna de 32 p.p.). |
| 4 | **Ausência de data augmentation** | Comprovado | Enunciado | Menor robustez a variações de posicionamento, exposição e equipamento; reforça o sobreajuste. |
| 5 | **Learning rate fixo (SGD 0,01)** sem busca nem agenda | Comprovado | Enunciado | Convergência subótima ou instável; nenhuma evidência de que 0,01 e 15 épocas sejam adequados. Sem early stopping, a época final pode ser pior que épocas anteriores. |
| 6 | **Uso exclusivo da acurácia global** | Comprovado | Enunciado | Esconde o desempenho por classe. Um modelo com recall de COVID próximo de zero pode exibir acurácia razoável. É justamente o que a equipe clínica percebe. |
| 7 | **Ausência de conjunto de teste independente** | Comprovado | Somente treino/validação | Se a validação orientou decisões (época, hiperparâmetros), os 61% são uma estimativa **otimista** e não há medida imparcial do desempenho. |
| 8 | **Vazamento de dados entre pacientes** | Risco | Divisão por imagem, sem ID de paciente | Imagens do mesmo paciente/exame em treino e validação inflacionam as métricas de validação; o desempenho real em pacientes novos seria menor. |
| 9 | **Atalhos visuais** (marcadores, texto, bordas, tipo de equipamento, fonte da imagem) | Risco | As classes do dataset vêm de repositórios distintos (ver tabela de fontes na Seção 3) | O modelo pode "acertar" pela origem da imagem e não pela doença ⇒ falha silenciosa ao mudar de hospital. |
| 10 | **Ausência de validação externa** (outros hospitais, equipamentos e populações) | Comprovado | Uma única base | Generalização desconhecida; risco de queda de desempenho e de vieses populacionais (idade, comorbidades, gravidade, protocolo de aquisição). |
| 11 | Ausência de calibração e análise de limiar | Risco | Decisão por *argmax* implícita | Probabilidades não interpretáveis como risco; não há como ajustar a sensibilidade ao contexto clínico (triagem × confirmação). |

### Consequências clínicas

- **Falsos negativos de COVID-19** (problemas 1, 2, 6): pacientes infectados classificados como Normal ou Viral Pneumonia podem deixar de ser isolados ou tratados.
- **Baixa generalização** (3, 4, 9, 10): o modelo pode falhar em um hospital diferente sem qualquer aviso.
- **Estimativas otimistas** (7, 8): decisões de adoção seriam tomadas com base em números inflados.
- **Impacto em decisões clínicas**: confiança indevida em uma ferramenta que, na classe de maior interesse, pode estar errando a maioria dos casos.

> A queixa "o modelo parece ignorar COVID-19" é o comportamento esperado da combinação dos problemas 1, 3 e 6: o classificador minimiza o erro médio favorecendo a classe majoritária, e a acurácia global não revela isso.

## 7. Preparação dos dados e métricas

### Transformações

- **Validação (determinística):** redimensionar para 224×224 → replicar o canal cinza em 3 canais → normalizar com média/desvio do ImageNet (exigido pelos pesos pré-treinados).
- **Treino:** as mesmas etapas, mais uma transformação afim **moderada** (rotação ±7°, translação ±4%, escala 0,95–1,05). **Não** usamos espelhamento horizontal: inverter lateralidade cria anatomia implausível (coração à direita) e pode inverter marcadores L/R. Também não alteramos contraste de forma agressiva, para não simular ou apagar opacidades.

### Métrica primária: recall da classe COVID-19

$$\text{Recall}_{COVID} = \frac{TP}{TP + FN}$$

O recall mede a **proporção de casos positivos de COVID-19 corretamente identificados**. Recall baixo significa que muitos casos positivos não estão sendo detectados (falsos negativos). Um aumento de recall **não** é automaticamente benéfico: se vier acompanhado de queda importante de precisão (mais falsos positivos — pacientes sem COVID sinalizados como positivos) ou de piora no macro F1 e nas outras classes, o modelo pode estar apenas "chutando COVID" com mais frequência. Por isso reportamos sempre recall, precisão, F1 por classe, macro F1 e matriz de confusão.

Para economizar I/O, as 1.200 imagens reais (≈100 MB em tons de cinza) são carregadas uma única vez na memória.

In [ ]:
IMAGE_CACHE = {}


def preload_images(paths) -> None:
    """Carrega imagens em tons de cinza (modo 'L') na memória, antes de criar os DataLoaders."""
    for path in tqdm([p for p in paths if p not in IMAGE_CACHE], desc='Carregando imagens'):
        with Image.open(path) as image:
            IMAGE_CACHE[path] = image.convert('L').copy()


class ChestXrayDataset(Dataset):
    """Dataset que lê imagens do cache e aplica as transformações."""

    def __init__(self, frame: pd.DataFrame, transform):
        self.paths = frame['path'].tolist()
        self.labels = frame['label'].astype(int).tolist()
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        image = IMAGE_CACHE[self.paths[index]]
        return self.transform(image), self.labels[index]


train_transform = T.Compose([
    T.Resize((CLS_IMAGE_SIZE, CLS_IMAGE_SIZE)),
    T.RandomAffine(degrees=7, translate=(0.04, 0.04), scale=(0.95, 1.05), fill=0),
    T.Grayscale(num_output_channels=3),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_transform = T.Compose([
    T.Resize((CLS_IMAGE_SIZE, CLS_IMAGE_SIZE)),
    T.Grayscale(num_output_channels=3),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


def make_loader(frame: pd.DataFrame, transform, shuffle: bool, seed: int = SEED, batch_size: int = CLS_BATCH_SIZE):
    generator = torch.Generator()
    generator.manual_seed(seed)
    return DataLoader(ChestXrayDataset(frame, transform), batch_size=batch_size, shuffle=shuffle,
                      num_workers=NUM_WORKERS, worker_init_fn=seed_worker, generator=generator,
                      pin_memory=torch.cuda.is_available(),
                      multiprocessing_context=LOADER_MP_CONTEXT if NUM_WORKERS > 0 else None)


with timer('preload_imagens_reais'):
    preload_images(sample_df['path'])
print(f'Imagens em cache: {len(IMAGE_CACHE)}')

In [ ]:
def compute_metrics(y_true, y_pred, class_names=CLASS_NAMES, positive_idx=COVID_IDX) -> dict:
    """Calcula métricas globais, por classe e da classe positiva (COVID-19)."""
    labels = list(range(len(class_names)))
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=labels, zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    tp = int(cm[positive_idx, positive_idx])
    fn = int(cm[positive_idx].sum() - tp)
    fp = int(cm[:, positive_idx].sum() - tp)
    return {
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'macro_f1': float(f1_score(y_true, y_pred, labels=labels, average='macro', zero_division=0)),
        'per_class': {name: {'precision': float(precision[i]), 'recall': float(recall[i]),
                             'f1': float(f1[i]), 'support': int(support[i])}
                      for i, name in enumerate(class_names)},
        'covid_recall': tp / (tp + fn) if (tp + fn) else 0.0,      # TP / (TP + FN)
        'covid_precision': tp / (tp + fp) if (tp + fp) else 0.0,   # 0 se nenhum caso foi previsto como COVID
        'covid_f1': float(f1[positive_idx]),
        'covid_tp': tp, 'covid_fn': fn, 'covid_fp': fp,
        'confusion_matrix': cm.tolist(),
        'report': classification_report(y_true, y_pred, labels=labels, target_names=class_names,
                                        zero_division=0, digits=3),
    }


def show_metrics(metrics: dict, title: str) -> None:
    """Exibe as métricas de forma legível."""
    print(f'===== {title} =====')
    print(f"Acurácia global : {metrics['accuracy']:.3f}")
    print(f"Macro F1        : {metrics['macro_f1']:.3f}")
    print(f"COVID-19 -> recall {metrics['covid_recall']:.3f} | precisão {metrics['covid_precision']:.3f} | "
          f"F1 {metrics['covid_f1']:.3f} | TP={metrics['covid_tp']} FN={metrics['covid_fn']} FP={metrics['covid_fp']}")
    display(pd.DataFrame(metrics['per_class']).T.round(3))
    print(metrics['report'])


def plot_confusion(cm, title: str, ax=None) -> None:
    """Matriz de confusão com contagens e percentual por linha (recall de cada classe)."""
    cm = np.asarray(cm)
    row_pct = cm / cm.sum(axis=1, keepdims=True).clip(min=1)
    annot = np.array([[f'{cm[i, j]}\n({100 * row_pct[i, j]:.0f}%)' for j in range(cm.shape[1])]
                      for i in range(cm.shape[0])])
    if ax is None:
        _, ax = plt.subplots(figsize=(5, 4.2))
    sns.heatmap(row_pct, annot=annot, fmt='', cmap='Blues', vmin=0, vmax=1, cbar=True,
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax,
                cbar_kws={'label': 'Proporção da classe real'})
    ax.set_title(title)
    ax.set_xlabel('Classe prevista')
    ax.set_ylabel('Classe real')


def plot_history(history: pd.DataFrame, title: str, best_epoch: int = None) -> None:
    """Curvas de loss, acurácia e métricas da validação interna por época."""
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    axes[0].plot(history['epoch'], history['train_loss'], 'o-', label='Treino')
    axes[0].plot(history['epoch'], history['internal_val_loss'], 's-', label='Validação interna')
    axes[0].set_title('Loss (entropia cruzada)')
    axes[0].set_ylabel('Loss')
    axes[1].plot(history['epoch'], history['train_acc'], 'o-', label='Treino')
    axes[1].plot(history['epoch'], history['internal_val_acc'], 's-', label='Validação interna')
    axes[1].set_title('Acurácia')
    axes[1].set_ylabel('Acurácia')
    axes[2].plot(history['epoch'], history['internal_val_macro_f1'], 's-', label='Macro F1 (val. interna)')
    axes[2].plot(history['epoch'], history['internal_val_covid_recall'], '^-', label='Recall COVID (val. interna)')
    axes[2].set_title('Métricas da validação interna')
    axes[2].set_ylabel('Valor')
    for ax in axes:
        ax.set_xlabel('Época')
        if best_epoch is not None:
            ax.axvline(best_epoch, color='gray', linestyle=':', label=f'checkpoint (época {best_epoch})')
        ax.legend(fontsize=8)
    fig.suptitle(title, fontsize=13)
    plt.tight_layout()

## 8. Treinamento do baseline

### Configuração

| Item | Cenário original | Baseline deste notebook |
|---|---|---|
| Arquitetura | ResNet-18 do zero | ResNet-18 **pré-treinada no ImageNet** (`IMAGENET1K_V1`) |
| Estratégia | Treinamento completo | **Extração de características**: backbone congelado, apenas a nova camada `fc` (512→3) é treinada |
| Entrada | n/d | 224×224, cinza replicado em 3 canais, normalização ImageNet |
| Divisão | 80/20 não estratificada | 80/20 estratificada e agrupada + validação interna |
| Augmentation | Nenhuma | Afim moderada (sem espelhamento) |
| Otimizador | SGD, lr fixo 0,01 | Adam, lr 1e-3, weight decay 1e-4, decaimento cosseno |
| Épocas | 15 | 15 |
| Perda | n/d | Entropia cruzada (multiclasse), **sem** pesos de classe |
| Seleção do modelo | Última época | Melhor **macro F1 na validação interna** (critério definido antes do treino) |
| Avaliação | Acurácia | Acurácia, precisão/recall/F1 por classe, macro F1, matriz de confusão |

> Este baseline é um **experimento mais bem controlado**, mas **não** é idêntico à ResNet-18 treinada do zero do cenário. Seus resultados não devem ser comparados diretamente aos 93%/61% do enunciado.

Escolhas deliberadas para isolar o efeito das imagens sintéticas:

- **Sem class weights e sem amostragem balanceada**: se esses mecanismos fossem usados, ficaria difícil separar o efeito deles do efeito da GAN. Eles aparecem no plano de melhorias como alternativas experimentais.
- **Macro F1** (e não o recall de COVID) foi escolhido para a seleção do checkpoint, para evitar privilegiar uma época que apenas super-prevê COVID.
- As camadas de BatchNorm do backbone ficam em modo de avaliação (estatísticas do ImageNet), consistente com extração de características.

O baseline é exatamente a configuração do **Experimento A**; ele é treinado nesta seção e seus resultados são analisados em detalhe na Seção 11.

In [ ]:
def build_classifier(num_classes: int = len(CLASS_NAMES), freeze_backbone: bool = True) -> nn.Module:
    """ResNet-18 pré-treinada no ImageNet com nova camada final de num_classes saídas."""
    model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False
    model.fc = nn.Linear(model.fc.in_features, num_classes)   # nova camada (treinável)
    return model


def set_train_mode(model: nn.Module) -> None:
    """Modo treino apenas para a cabeça: BatchNorm do backbone congelado permanece em eval."""
    model.train()
    for module in model.modules():
        if isinstance(module, nn.BatchNorm2d) and not any(p.requires_grad for p in module.parameters()):
            module.eval()


def train_one_epoch(model, loader, criterion, optimizer):
    set_train_mode(model)
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * labels.size(0)
        correct += (logits.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def predict(model, loader, criterion=None):
    """Retorna rótulos reais, previstos, probabilidades e loss média."""
    model.eval()
    y_true, y_pred, probs, total_loss = [], [], [], 0.0
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        logits = model(images)
        if criterion is not None:
            total_loss += criterion(logits, labels).item() * labels.size(0)
        probs.append(F.softmax(logits, dim=1).cpu())
        y_true.append(labels.cpu())
        y_pred.append(logits.argmax(1).cpu())
    y_true = torch.cat(y_true).numpy()
    return y_true, torch.cat(y_pred).numpy(), torch.cat(probs).numpy(), total_loss / max(len(y_true), 1)


def fit_classifier(train_frame: pd.DataFrame, internal_val_frame: pd.DataFrame, run_name: str, seed: int = SEED):
    """Treina a cabeça da ResNet-18 e seleciona o checkpoint pela macro F1 da validação interna."""
    set_seed(seed)                                  # mesma inicialização da camada fc em todos os experimentos
    model = build_classifier().to(DEVICE)
    init_fingerprint = float(model.fc.weight.detach().sum().cpu())
    trainable = [p for p in model.parameters() if p.requires_grad]
    print(f'[{run_name}] parâmetros treináveis: {sum(p.numel() for p in trainable):,} | '
          f'impressão digital da fc inicial: {init_fingerprint:.6f}')

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(trainable, lr=CLS_LR, weight_decay=CLS_WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CLS_EPOCHS)
    train_loader = make_loader(train_frame, train_transform, shuffle=True, seed=seed)
    internal_loader = make_loader(internal_val_frame, eval_transform, shuffle=False, seed=seed)

    history, best_score, best_epoch, best_state = [], -np.inf, None, None
    for epoch in range(1, CLS_EPOCHS + 1):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer)
        y_true, y_pred, _, iv_loss = predict(model, internal_loader, criterion)
        iv_metrics = compute_metrics(y_true, y_pred)
        row = {'epoch': epoch, 'lr': optimizer.param_groups[0]['lr'], 'train_loss': train_loss,
               'train_acc': train_acc, 'internal_val_loss': iv_loss, 'internal_val_acc': iv_metrics['accuracy'],
               'internal_val_macro_f1': iv_metrics['macro_f1'],
               'internal_val_covid_recall': iv_metrics['covid_recall']}
        history.append(row)
        scheduler.step()
        if row[CHECKPOINT_METRIC] > best_score:      # critério fixado antes do treino
            best_score, best_epoch = row[CHECKPOINT_METRIC], epoch
            best_state = copy.deepcopy(model.state_dict())
        print(f"[{run_name}] época {epoch:02d} | loss {train_loss:.4f}/{iv_loss:.4f} | "
              f"acc {train_acc:.3f}/{iv_metrics['accuracy']:.3f} | macroF1(int) {iv_metrics['macro_f1']:.3f} | "
              f"recall COVID(int) {iv_metrics['covid_recall']:.3f}")

    model.load_state_dict(best_state)
    torch.save(best_state, CHECKPOINT_DIR / f'{run_name}_best.pt')
    print(f'[{run_name}] melhor época: {best_epoch} ({CHECKPOINT_METRIC} = {best_score:.3f})')
    return model, pd.DataFrame(history), best_epoch, init_fingerprint


def run_experiment(run_name: str, train_frame: pd.DataFrame, description: str) -> dict:
    """Treina, avalia na validação real e salva histórico, previsões e métricas."""
    start = time.perf_counter()
    model, history, best_epoch, fingerprint = fit_classifier(train_frame, internal_val_df, run_name)
    y_true, y_pred, probs, val_loss = predict(model, make_loader(val_df, eval_transform, shuffle=False),
                                              nn.CrossEntropyLoss())
    metrics = compute_metrics(y_true, y_pred)
    elapsed = time.perf_counter() - start
    EXEC_TIMES[f'experimento_{run_name}'] = elapsed

    history.to_csv(RESULTS_DIR / f'history_{run_name}.csv', index=False)
    predictions = val_df[['path', 'class_name', 'label']].copy()
    predictions['pred'] = y_pred
    for i, name in enumerate(CLASS_NAMES):
        predictions[f'prob_{name}'] = probs[:, i]
    predictions.to_csv(RESULTS_DIR / f'predictions_{run_name}.csv', index=False)
    result = {'run_name': run_name, 'description': description, 'best_epoch': best_epoch,
              'checkpoint_metric': CHECKPOINT_METRIC, 'fc_init_fingerprint': fingerprint,
              'train_counts': train_frame['class_name'].value_counts().reindex(CLASS_NAMES, fill_value=0).to_dict(),
              'n_synthetic': int(train_frame['is_synthetic'].sum()), 'val_loss': val_loss,
              'elapsed_seconds': elapsed, 'metrics': {k: v for k, v in metrics.items() if k != 'report'},
              'report': metrics['report'], 'quick_run': QUICK_RUN, 'seed': SEED}
    save_json(result, RESULTS_DIR / f'results_{run_name}.json')
    print(f'⏱  experimento {run_name}: {elapsed / 60:.1f} min — resultados salvos em {RESULTS_DIR}')
    return result

In [ ]:
# Verificação visual do pipeline: um lote de treino após as transformações (desnormalizado)
preview_images, preview_labels = next(iter(make_loader(train_fit_df, train_transform, shuffle=True)))
print(f'Formato do lote: {tuple(preview_images.shape)} (lote, canais, altura, largura)')
mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
fig, axes = plt.subplots(2, 6, figsize=(14, 5))
for ax, image, label in zip(axes.ravel(), preview_images, preview_labels):
    ax.imshow((image * std + mean).clamp(0, 1).permute(1, 2, 0)[..., 0], cmap='gray')
    ax.set_title(CLASS_NAMES[label], fontsize=9)
    ax.axis('off')
fig.suptitle('Lote de treino após augmentation moderada (entrada real do modelo)', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Treinamento do baseline = Experimento A (somente imagens reais de treino)
train_fit_df['is_synthetic'] = False
internal_val_df['is_synthetic'] = False
val_df['is_synthetic'] = False

results_A = run_experiment('A', train_fit_df, 'Somente imagens reais de treino (baseline)')

## 9. Treinamento da GAN condicional (cGAN)

### Dados

- **Somente** o conjunto *treino de ajuste* (imagens reais) é usado. A validação real e a validação interna **não** participam (há uma verificação explícita abaixo).
- Imagens em tons de cinza redimensionadas para **64×64** e normalizadas para [-1, 1].
- A cGAN é treinada com as três classes (é isso que permite condicionar a geração), mas com **amostragem balanceada por classe** (`WeightedRandomSampler`): sem isso, a classe COVID-19 — com menos de 100 imagens de treino — apareceria em apenas ~10% dos lotes.

### Arquitetura (estilo DCGAN condicional)

**Gerador** — entrada: vetor latente `z` (100) + embedding da classe (50):
`Linear(150 → 512·4·4) + BN + ReLU` → `ConvT 512→256` (8×8) → `ConvT 256→128` (16×16) → `ConvT 128→64` (32×32) → `ConvT 64→1` (64×64) + `Tanh`. Cada ConvT intermediária usa BatchNorm + ReLU.

**Discriminador** — entrada: imagem (1×64×64) concatenada a um **mapa de classe** (embedding da classe projetado para 1×64×64):
`Conv 2→64` (32×32) → `Conv 64→128` (16×16) → `Conv 128→256` (8×8) → `Conv 256→512` (4×4) → `Linear(8192 → 1)`. LeakyReLU(0,2) e **normalização espectral** em todas as camadas (estabiliza o treino sem BatchNorm no discriminador).

### Treinamento

| Item | Valor |
|---|---|
| Loss | Entropia cruzada binária com logits (GAN não saturante) |
| Label smoothing | rótulo real = 0,9 (apenas no discriminador) |
| Otimizador | Adam, lr = 2e-4, β = (0,5; 0,999) para G e D |
| Lote | 64 |
| Épocas | `GAN_EPOCHS` (padrão 400 ≈ 5 mil iterações; poucos minutos em uma T4) |
| Monitoramento | Grade fixa de amostras (mesmo ruído) exibida `GAN_N_PREVIEWS` vezes e salva em `gan_samples/` |

In [ ]:
# Garantia: nenhuma imagem de validação (real ou interna) entra no treino da GAN
gan_train_df = train_fit_df.copy()
forbidden = set(val_df['path']) | set(internal_val_df['path'])
assert not (set(gan_train_df['path']) & forbidden), 'Imagem de validação no treino da GAN!'
assert not (set(gan_train_df['md5']) & (set(val_df['md5']) | set(internal_val_df['md5'])))
print('Imagens reais usadas para treinar a cGAN:')
display(gan_train_df['class_name'].value_counts().reindex(CLASS_NAMES).to_frame('n'))

gan_transform = T.Compose([
    T.Resize((GAN_IMAGE_SIZE, GAN_IMAGE_SIZE)),
    T.ToTensor(),                        # 1 canal, [0, 1]
    T.Normalize([0.5], [0.5]),           # [-1, 1], compatível com Tanh do gerador
])

class_counts = gan_train_df['label'].value_counts().to_dict()
sample_weights = gan_train_df['label'].map(lambda y: 1.0 / class_counts[y]).values
gan_generator = torch.Generator()
gan_generator.manual_seed(SEED)
gan_sampler = WeightedRandomSampler(torch.tensor(np.asarray(sample_weights, dtype=np.float64)),
                                    num_samples=len(gan_train_df), replacement=True, generator=gan_generator)
gan_loader = DataLoader(ChestXrayDataset(gan_train_df, gan_transform), batch_size=GAN_BATCH_SIZE,
                        sampler=gan_sampler, drop_last=True, num_workers=NUM_WORKERS,
                        worker_init_fn=seed_worker, generator=gan_generator,
                        multiprocessing_context=LOADER_MP_CONTEXT if NUM_WORKERS > 0 else None)
print(f'Iterações por época: {len(gan_loader)} | total previsto: {len(gan_loader) * GAN_EPOCHS}')

In [ ]:
class ConditionalGenerator(nn.Module):
    def __init__(self, z_dim=GAN_Z_DIM, n_classes=len(CLASS_NAMES), emb_dim=GAN_EMB_DIM, ngf=64):
        super().__init__()
        self.label_embedding = nn.Embedding(n_classes, emb_dim)
        self.project = nn.Sequential(
            nn.Linear(z_dim + emb_dim, ngf * 8 * 4 * 4, bias=False),
            nn.BatchNorm1d(ngf * 8 * 4 * 4),
            nn.ReLU(True),
        )

        def up(in_ch, out_ch):
            return [nn.ConvTranspose2d(in_ch, out_ch, 4, 2, 1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(True)]

        self.net = nn.Sequential(
            *up(ngf * 8, ngf * 4),      # 8×8
            *up(ngf * 4, ngf * 2),      # 16×16
            *up(ngf * 2, ngf),          # 32×32
            nn.ConvTranspose2d(ngf, 1, 4, 2, 1),   # 64×64
            nn.Tanh(),
        )
        self.ngf = ngf

    def forward(self, z, labels):
        x = torch.cat([z, self.label_embedding(labels)], dim=1)
        x = self.project(x).view(-1, self.ngf * 8, 4, 4)
        return self.net(x)


class ConditionalDiscriminator(nn.Module):
    def __init__(self, n_classes=len(CLASS_NAMES), ndf=64, image_size=GAN_IMAGE_SIZE):
        super().__init__()
        sn = nn.utils.spectral_norm
        self.image_size = image_size
        self.label_map = nn.Embedding(n_classes, image_size * image_size)   # classe -> mapa 64×64
        self.net = nn.Sequential(
            sn(nn.Conv2d(2, ndf, 4, 2, 1)), nn.LeakyReLU(0.2, True),            # 32×32
            sn(nn.Conv2d(ndf, ndf * 2, 4, 2, 1)), nn.LeakyReLU(0.2, True),      # 16×16
            sn(nn.Conv2d(ndf * 2, ndf * 4, 4, 2, 1)), nn.LeakyReLU(0.2, True),  # 8×8
            sn(nn.Conv2d(ndf * 4, ndf * 8, 4, 2, 1)), nn.LeakyReLU(0.2, True),  # 4×4
            nn.Flatten(),
            sn(nn.Linear(ndf * 8 * 4 * 4, 1)),
        )

    def forward(self, images, labels):
        label_map = self.label_map(labels).view(-1, 1, self.image_size, self.image_size)
        return self.net(torch.cat([images, label_map], dim=1))


def init_dcgan_weights(module):
    """Inicialização recomendada no artigo DCGAN (Radford et al., 2016)."""
    if isinstance(module, (nn.Conv2d, nn.ConvTranspose2d, nn.Linear)):
        nn.init.normal_(module.weight, 0.0, 0.02)
        if module.bias is not None:
            nn.init.zeros_(module.bias)
    elif isinstance(module, (nn.BatchNorm2d, nn.BatchNorm1d)):
        nn.init.normal_(module.weight, 1.0, 0.02)
        nn.init.zeros_(module.bias)


def show_gan_grid(generator, fixed_z, fixed_labels, title, path=None, display_figure=True):
    """Gera uma grade (linhas = classes) com ruído fixo para acompanhar a evolução."""
    generator.eval()
    with torch.no_grad():
        fake = generator(fixed_z, fixed_labels).cpu() * 0.5 + 0.5
    generator.train()
    grid = make_grid(fake, nrow=fixed_z.size(0) // len(CLASS_NAMES), padding=2, pad_value=1)
    if path is not None:
        save_image(grid, path)
    if display_figure:
        plt.figure(figsize=(12, 4.6))
        plt.imshow(grid.permute(1, 2, 0)[..., 0], cmap='gray', vmin=0, vmax=1)
        plt.yticks([(GAN_IMAGE_SIZE + 2) * (i + 0.5) for i in range(len(CLASS_NAMES))], CLASS_NAMES)
        plt.xticks([])
        plt.title(title)
        plt.show()

In [ ]:
def train_cgan(loader, epochs: int = GAN_EPOCHS, seed: int = SEED):
    """Treina a cGAN e retorna gerador, discriminador e histórico de losses."""
    set_seed(seed)
    generator = ConditionalGenerator().to(DEVICE)
    discriminator = ConditionalDiscriminator().to(DEVICE)
    generator.apply(init_dcgan_weights)
    discriminator.apply(init_dcgan_weights)
    opt_g = torch.optim.Adam(generator.parameters(), lr=GAN_LR, betas=GAN_BETAS)
    opt_d = torch.optim.Adam(discriminator.parameters(), lr=GAN_LR, betas=GAN_BETAS)
    bce = nn.BCEWithLogitsLoss()

    n_cols = 10
    fixed_labels = torch.arange(len(CLASS_NAMES)).repeat_interleave(n_cols).to(DEVICE)
    fixed_z = torch.randn(len(fixed_labels), GAN_Z_DIM, device=DEVICE)
    preview_epochs = set(np.linspace(1, epochs, num=min(GAN_N_PREVIEWS, epochs), dtype=int).tolist())
    history = []

    for epoch in range(1, epochs + 1):
        sums = {'loss_d': 0.0, 'loss_g': 0.0, 'd_real': 0.0, 'd_fake': 0.0}
        for real, labels in loader:
            real, labels = real.to(DEVICE), labels.to(DEVICE)
            batch = real.size(0)
            z = torch.randn(batch, GAN_Z_DIM, device=DEVICE)
            fake = generator(z, labels)                 # mesmas classes do lote real (balanceado)

            # --- Discriminador: real -> 0,9 ; falso -> 0
            logits_real = discriminator(real, labels)
            logits_fake = discriminator(fake.detach(), labels)
            loss_d = (bce(logits_real, torch.full_like(logits_real, GAN_REAL_LABEL)) +
                      bce(logits_fake, torch.zeros_like(logits_fake)))
            opt_d.zero_grad(set_to_none=True)
            loss_d.backward()
            opt_d.step()

            # --- Gerador: tenta fazer o discriminador classificar o falso como real
            logits_fake_g = discriminator(fake, labels)
            loss_g = bce(logits_fake_g, torch.ones_like(logits_fake_g))
            opt_g.zero_grad(set_to_none=True)
            loss_g.backward()
            opt_g.step()

            sums['loss_d'] += loss_d.item()
            sums['loss_g'] += loss_g.item()
            sums['d_real'] += torch.sigmoid(logits_real).mean().item()
            sums['d_fake'] += torch.sigmoid(logits_fake).mean().item()
        history.append({'epoch': epoch, **{k: v / len(loader) for k, v in sums.items()}})

        if epoch in preview_epochs:
            row = history[-1]
            print(f"época {epoch:4d}/{epochs} | loss D {row['loss_d']:.3f} | loss G {row['loss_g']:.3f} | "
                  f"D(real) {row['d_real']:.2f} | D(falso) {row['d_fake']:.2f}")
            show_gan_grid(generator, fixed_z, fixed_labels, f'Amostras da cGAN — época {epoch}',
                          GAN_SAMPLES_DIR / f'epoch_{epoch:04d}.png')

    torch.save(generator.state_dict(), CHECKPOINT_DIR / 'cgan_generator.pt')
    torch.save(discriminator.state_dict(), CHECKPOINT_DIR / 'cgan_discriminator.pt')
    return generator, discriminator, pd.DataFrame(history)


preload_images(gan_train_df['path'])
with timer('treino_cgan'):
    generator, discriminator, gan_history = train_cgan(gan_loader)
gan_history.to_csv(RESULTS_DIR / 'gan_history.csv', index=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(gan_history['epoch'], gan_history['loss_d'], label='Discriminador')
axes[0].plot(gan_history['epoch'], gan_history['loss_g'], label='Gerador')
axes[0].set_title('Loss da cGAN por época')
axes[0].set_ylabel('Loss (BCE)')
axes[1].plot(gan_history['epoch'], gan_history['d_real'], label='D(real)')
axes[1].plot(gan_history['epoch'], gan_history['d_fake'], label='D(falso)')
axes[1].set_title('Saída média do discriminador')
axes[1].set_ylabel('Probabilidade de "real"')
for ax in axes:
    ax.set_xlabel('Época')
    ax.legend()
plt.tight_layout()
save_fig('cgan_losses')
plt.show()

In [ ]:
def generate_synthetic_images(generator, n_images: int, class_name: str = 'COVID', seed: int = SEED,
                              out_dir: Path = SYNTHETIC_DIR) -> pd.DataFrame:
    """Gera n_images sintéticas da classe pedida, salva em PNG (64×64) e retorna um DataFrame."""
    for old in out_dir.glob('*.png'):          # evita misturar execuções anteriores
        old.unlink()
    if n_images <= 0:
        return pd.DataFrame(columns=['path', 'file_name', 'class_name', 'label', 'is_synthetic'])
    torch.manual_seed(seed)
    generator.eval()
    records, label = [], CLASS_TO_IDX[class_name]
    with torch.no_grad():
        for start in range(0, n_images, 64):
            batch = min(64, n_images - start)
            z = torch.randn(batch, GAN_Z_DIM, device=DEVICE)
            labels = torch.full((batch,), label, dtype=torch.long, device=DEVICE)
            images = (generator(z, labels).cpu() * 0.5 + 0.5).clamp(0, 1)
            for offset, image in enumerate(images):
                path = out_dir / f'synthetic_{class_name.lower()}_{start + offset:04d}.png'
                Image.fromarray((image[0].numpy() * 255).round().astype(np.uint8), mode='L').save(path)
                records.append({'path': str(path), 'file_name': path.name, 'class_name': class_name,
                                'label': label, 'is_synthetic': True})
    return pd.DataFrame(records)


with timer('geracao_sinteticas'):
    synthetic_df = generate_synthetic_images(generator, N_SYNTHETIC)
synthetic_df['md5'] = synthetic_df['path'].map(lambda p: hashlib.md5(Path(p).read_bytes()).hexdigest())
synthetic_df['group_id'] = -1
synthetic_df['source'] = 'sintetica_cgan'
preload_images(synthetic_df['path'])
print(f'{len(synthetic_df)} imagens sintéticas de COVID-19 salvas em {SYNTHETIC_DIR}')

## 10. Inspeção das imagens sintéticas

A inspeção combina uma **grade visual** (sintéticas × reais de treino na mesma resolução 64×64) com **indicadores quantitativos simples**:

- **Diversidade:** distância média entre pares de imagens sintéticas, comparada à mesma medida entre imagens reais de COVID-19. Razão muito abaixo de 1 sugere **colapso de modos** (pouca variedade).
- **Memorização:** distância de cada sintética até a imagem real de treino mais próxima, comparada à distância típica entre imagens reais distintas. Valores muito pequenos sugerem que a GAN está **copiando** exemplos de treino.
- **Quase-duplicatas internas:** pares de sintéticas com hash perceptual quase idêntico.
- **Histograma de intensidades:** diferenças na distribuição de brilho/contraste.

Esses indicadores são heurísticos: **não** medem plausibilidade clínica e **não** substituem a revisão por radiologistas.

In [ ]:
def to_gan_array(paths) -> torch.Tensor:
    """Converte imagens para tensores 64×64 em [0, 1] (mesma resolução das sintéticas)."""
    resize = T.Compose([T.Resize((GAN_IMAGE_SIZE, GAN_IMAGE_SIZE)), T.ToTensor()])
    return torch.stack([resize(IMAGE_CACHE[p]) for p in paths])


real_covid_train = to_gan_array(gan_train_df.loc[gan_train_df['class_name'] == 'COVID', 'path'])
real_all_train = to_gan_array(gan_train_df['path'])
synthetic_tensor = to_gan_array(synthetic_df['path'])

n_show = 16
fig, axes = plt.subplots(2, 1, figsize=(14, 4.6))
for ax, tensor, title in [(axes[0], synthetic_tensor[:n_show], f'Sintéticas COVID-19 (cGAN), 64×64 — {n_show} de {len(synthetic_tensor)}'),
                          (axes[1], real_covid_train[:n_show], 'Reais COVID-19 do treino, reduzidas para 64×64')]:
    ax.imshow(make_grid(tensor, nrow=n_show, padding=2, pad_value=1).permute(1, 2, 0)[..., 0], cmap='gray')
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout()
save_fig('sinteticas_vs_reais')
plt.show()

In [ ]:
def mean_pairwise_distance(x: torch.Tensor) -> float:
    flat = x.flatten(1)
    dist = torch.cdist(flat, flat)
    n = len(flat)
    return float(dist.sum() / (n * (n - 1)))


def nearest_neighbor_distance(query: torch.Tensor, reference: torch.Tensor, exclude_self: bool = False) -> torch.Tensor:
    dist = torch.cdist(query.flatten(1), reference.flatten(1))
    if exclude_self:
        dist.fill_diagonal_(float('inf'))
    return dist.min(dim=1)


diversity_synth = mean_pairwise_distance(synthetic_tensor)
diversity_real = mean_pairwise_distance(real_covid_train)
nn_synth = nearest_neighbor_distance(synthetic_tensor, real_all_train)
nn_real = nearest_neighbor_distance(real_covid_train, real_covid_train, exclude_self=True)
synth_bits = np.stack([dhash_bits(p) for p in synthetic_df['path']])
synth_ham = hamming_matrix(synth_bits, synth_bits)
np.fill_diagonal(synth_ham, 999)
n_synth_near_dups = int(np.triu(synth_ham <= DHASH_MAX_DISTANCE, k=1).sum())

gan_diagnostics = {
    'diversidade_sinteticas': diversity_synth,
    'diversidade_reais_covid': diversity_real,
    'razao_diversidade': diversity_synth / diversity_real,
    'nn_sintetica_para_real_mediana': float(nn_synth.values.median()),
    'nn_real_para_real_mediana': float(nn_real.values.median()),
    'razao_memorizacao': float(nn_synth.values.median() / nn_real.values.median()),
    'pares_sinteticos_quase_identicos': n_synth_near_dups,
    'intensidade_media_sintetica': float(synthetic_tensor.mean()),
    'intensidade_media_real_covid': float(real_covid_train.mean()),
    'desvio_intensidade_sintetica': float(synthetic_tensor.std()),
    'desvio_intensidade_real_covid': float(real_covid_train.std()),
}
save_json(gan_diagnostics, RESULTS_DIR / 'gan_diagnostics.json')
display(pd.Series(gan_diagnostics, name='valor').to_frame().round(4))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(real_covid_train.flatten().numpy(), bins=50, alpha=0.6, density=True, label='Reais COVID (treino)')
axes[0].hist(synthetic_tensor.flatten().numpy(), bins=50, alpha=0.6, density=True, label='Sintéticas COVID')
axes[0].set_title('Distribuição de intensidade dos pixels (64×64)')
axes[0].set_xlabel('Intensidade [0, 1]')
axes[0].set_ylabel('Densidade')
axes[0].legend()
axes[1].hist(nn_real.values.numpy(), bins=30, alpha=0.6, density=True, label='Real → real mais próxima')
axes[1].hist(nn_synth.values.numpy(), bins=30, alpha=0.6, density=True, label='Sintética → real mais próxima')
axes[1].set_title('Distância ao vizinho mais próximo (L2, 64×64)')
axes[1].set_xlabel('Distância L2')
axes[1].set_ylabel('Densidade')
axes[1].legend()
plt.tight_layout()
save_fig('cgan_diagnosticos')
plt.show()

# Pares sintética × real mais próxima (inspeção de memorização)
closest = torch.argsort(nn_synth.values)[:6]
fig, axes = plt.subplots(2, 6, figsize=(14, 5))
for col, idx in enumerate(closest.tolist()):
    axes[0, col].imshow(synthetic_tensor[idx, 0], cmap='gray')
    axes[0, col].set_title(f'Sintética #{idx}\nd={nn_synth.values[idx]:.1f}', fontsize=8)
    axes[1, col].imshow(real_all_train[nn_synth.indices[idx], 0], cmap='gray')
    axes[1, col].set_title(f"Real mais próxima\n({gan_train_df.iloc[int(nn_synth.indices[idx])]['class_name']})", fontsize=8)
for ax in axes.ravel():
    ax.axis('off')
fig.suptitle('Sintéticas mais próximas de uma imagem real de treino', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
def describe_gan_quality(diag: dict) -> str:
    """Resumo automático e conservador dos indicadores (não substitui inspeção visual nem clínica)."""
    notes = []
    ratio = diag['razao_diversidade']
    if ratio < 0.6:
        notes.append(f'- Diversidade baixa (razão {ratio:.2f}): **forte indício de colapso de modos**.')
    elif ratio < 0.85:
        notes.append(f'- Diversidade reduzida em relação às reais (razão {ratio:.2f}): possível colapso parcial de modos.')
    else:
        notes.append(f'- Diversidade comparável à das reais (razão {ratio:.2f}); isso não garante variedade clinicamente relevante.')
    mem = diag['razao_memorizacao']
    if mem < 0.5:
        notes.append(f'- Sintéticas muito próximas de imagens de treino (razão {mem:.2f}): **risco de memorização/cópia**.')
    else:
        notes.append(f'- Distância às reais mais próximas comparável à distância entre reais (razão {mem:.2f}): sem sinal forte de cópia direta.')
    if diag['pares_sinteticos_quase_identicos'] > 0:
        notes.append(f"- {diag['pares_sinteticos_quase_identicos']} pares de sintéticas quase idênticos (dHash) — redundância.")
    delta_std = diag['desvio_intensidade_sintetica'] - diag['desvio_intensidade_real_covid']
    notes.append(f'- Diferença no desvio-padrão de intensidade (sintética − real): {delta_std:+.3f} '
                 '(valores negativos indicam imagens mais "lavadas"/borradas).')
    notes.append('- As sintéticas têm 64×64 pixels e são ampliadas para 224×224 no classificador: '
                 'ficam mais borradas que as reais (299×299). O classificador pode usar essa diferença como **atalho** '
                 '(“borrado ⇒ COVID”), o que não ajuda a reconhecer COVID em imagens reais.')
    return '\n'.join(notes)


display(Markdown('### Indicadores automáticos da cGAN (desta execução)\n' + describe_gan_quality(gan_diagnostics)))

### Roteiro de discussão da inspeção visual

Ao observar as grades acima (e as salvas em `gan_samples/`), avalie e registre:

1. **Semelhança visual:** as sintéticas reproduzem a forma geral do tórax (campos pulmonares escuros, mediastino e coração claros, clavículas, diafragma)? Em 64×64, detalhes finos como opacidades em vidro fosco, padrão intersticial e consolidações periféricas — justamente os achados associados à COVID-19 — **não** são representáveis com fidelidade.
2. **Diversidade:** variam posicionamento, tamanho do tórax, exposição e lateralidade dos achados, ou todas parecem variações mínimas de um mesmo "molde"? Compare com o indicador de diversidade.
3. **Artefatos:** padrões de tabuleiro (*checkerboard*) típicos de convoluções transpostas, bordas irregulares, costelas descontínuas, estruturas anatômicas duplicadas ou ausentes, textos/marcadores "derretidos".
4. **Colapso de modos:** amostras repetidas ou muito parecidas entre si, independentemente do ruído de entrada.
5. **Plausibilidade clínica:** padrões que não existem em radiografias reais podem ensinar o classificador a reconhecer **artefatos da GAN** como se fossem sinais de COVID-19.

> **As imagens sintéticas não são evidência clínica.** Parecer realista para um leigo (ou mesmo para métricas automáticas) não significa conter achados radiológicos corretos. Nenhuma imagem sintética foi revisada por radiologista neste trabalho. Se as amostras forem claramente inadequadas (ruído, colapso total, borrões sem anatomia), esse fato deve ser registrado e o Experimento B deve ser lido como teste de uma GAN **insuficientemente treinada**, não como avaliação definitiva da abordagem generativa.

## 11. Experimento A — sem imagens sintéticas

O baseline treinado na Seção 8 **é** o Experimento A: ResNet-18 pré-treinada, backbone congelado, somente imagens reais do *treino de ajuste*, checkpoint escolhido pela macro F1 da validação interna. Abaixo estão o histórico por época e as métricas finais na **validação real** (avaliada uma única vez, com o checkpoint escolhido). Os resultados foram salvos em `results_A.json`, `history_A.csv` e `predictions_A.csv` para reutilização.

In [ ]:
history_A = pd.read_csv(RESULTS_DIR / 'history_A.csv')
display(history_A.round(4))
plot_history(history_A, 'Experimento A — evolução do treinamento', results_A['best_epoch'])
save_fig('historico_A')
plt.show()

print('Exemplos de treino por classe (A):', results_A['train_counts'])
show_metrics({**results_A['metrics'], 'report': results_A['report']}, 'Experimento A — validação real')
plot_confusion(results_A['metrics']['confusion_matrix'], 'Experimento A — matriz de confusão (validação real)')
save_fig('confusao_A')
plt.show()
print(f"Tempo de execução do experimento A: {results_A['elapsed_seconds'] / 60:.1f} min")

## 12. Experimento B — com imagens sintéticas de COVID-19

Única diferença em relação ao Experimento A: o conjunto de treino recebe `N_SYNTHETIC` imagens sintéticas de COVID-19. Todo o resto é igual:

- mesmas imagens reais de treino, mesma validação interna, mesma validação real (somente imagens reais);
- nova instância da ResNet-18 com os **mesmos pesos ImageNet** e a **mesma semente** (logo, a mesma inicialização da camada `fc` — conferida pela "impressão digital" impressa no log);
- **não** reaproveita o checkpoint do Experimento A;
- mesmos hiperparâmetros, transformações e critério de checkpoint.

In [ ]:
train_B_df = pd.concat([train_fit_df, synthetic_df], ignore_index=True)

# Verificações de integridade do desenho experimental
assert not val_df['is_synthetic'].any() and not internal_val_df['is_synthetic'].any(), 'Sintética na validação!'
assert set(train_fit_df['path']) <= set(train_B_df['path']), 'Imagens reais de treino alteradas!'
assert len(train_B_df) - len(train_fit_df) == len(synthetic_df) == N_SYNTHETIC
assert not (set(synthetic_df['path']) & (set(val_df['path']) | set(internal_val_df['path'])))

composition = pd.DataFrame({
    'Reais (A e B)': train_fit_df['class_name'].value_counts().reindex(CLASS_NAMES, fill_value=0),
    'Sintéticas (só B)': synthetic_df['class_name'].value_counts().reindex(CLASS_NAMES, fill_value=0),
    'Total B': train_B_df['class_name'].value_counts().reindex(CLASS_NAMES, fill_value=0),
})
composition['% em B'] = (100 * composition['Total B'] / composition['Total B'].sum()).round(1)
display(composition)

results_B = run_experiment('B', train_B_df, f'Imagens reais de treino + {N_SYNTHETIC} sintéticas de COVID-19 (cGAN)')
assert abs(results_A['fc_init_fingerprint'] - results_B['fc_init_fingerprint']) < 1e-6, 'Inicializações diferentes!'
print('Inicialização da camada fc idêntica nos dois experimentos.')

In [ ]:
history_B = pd.read_csv(RESULTS_DIR / 'history_B.csv')
display(history_B.round(4))
plot_history(history_B, 'Experimento B — evolução do treinamento', results_B['best_epoch'])
save_fig('historico_B')
plt.show()

show_metrics({**results_B['metrics'], 'report': results_B['report']}, 'Experimento B — validação real')
plot_confusion(results_B['metrics']['confusion_matrix'], 'Experimento B — matriz de confusão (validação real)')
save_fig('confusao_B')
plt.show()
print(f"Tempo de execução do experimento B: {results_B['elapsed_seconds'] / 60:.1f} min")

## 13. Comparação e discussão dos resultados

Os resultados são relidos dos arquivos JSON salvos, de modo que esta seção pode ser reexecutada isoladamente.

- **Diferença absoluta** (em pontos percentuais, p.p.): `B − A`. Ex.: recall 0,50 → 0,60 é **+10 p.p.**
- **Diferença relativa** (%): `(B − A) / A`. Ex.: 0,50 → 0,60 é **+20%**. Não é definida quando A = 0.

Como a validação real tem poucos casos de COVID-19, estimamos também a **incerteza** por *bootstrap pareado* (2.000 reamostragens das mesmas imagens de validação para A e B) e o intervalo de Wilson de 95% para o recall de cada experimento.

In [ ]:
with open(RESULTS_DIR / 'results_A.json', encoding='utf-8') as file:
    res_A = json.load(file)
with open(RESULTS_DIR / 'results_B.json', encoding='utf-8') as file:
    res_B = json.load(file)

metric_keys = [('Acurácia global', 'accuracy'), ('Macro F1', 'macro_f1'), ('Recall COVID-19', 'covid_recall'),
               ('Precisão COVID-19', 'covid_precision'), ('F1-score COVID-19', 'covid_f1')]
rows = []
for label, key in metric_keys:
    a, b = res_A['metrics'][key], res_B['metrics'][key]
    rows.append({'Métrica': label, 'Experimento A': a, 'Experimento B': b,
                 'Diferença (p.p.)': 100 * (b - a),
                 'Diferença relativa (%)': 100 * (b - a) / a if a else np.nan})
for label, key in [('Falsos negativos COVID', 'covid_fn'), ('Falsos positivos COVID', 'covid_fp')]:
    a, b = res_A['metrics'][key], res_B['metrics'][key]
    rows.append({'Métrica': label, 'Experimento A': a, 'Experimento B': b,
                 'Diferença (p.p.)': np.nan, 'Diferença relativa (%)': np.nan, 'Diferença (contagem)': b - a})
comparison = pd.DataFrame(rows).set_index('Métrica')
comparison.to_csv(RESULTS_DIR / 'comparacao_A_B.csv')
display(comparison.style.format(precision=3, na_rep='—'))

per_class = pd.concat({'A': pd.DataFrame(res_A['metrics']['per_class']).T,
                       'B': pd.DataFrame(res_B['metrics']['per_class']).T}, axis=1).round(3)
print('Métricas por classe (validação real):')
display(per_class)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
plot_confusion(res_A['metrics']['confusion_matrix'], 'Experimento A (somente reais)', axes[0])
plot_confusion(res_B['metrics']['confusion_matrix'], f"Experimento B (+{res_B['n_synthetic']} sintéticas)", axes[1])
plt.tight_layout()
save_fig('confusao_A_vs_B')
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for history, name, style in [(history_A, 'A', '-'), (history_B, 'B', '--')]:
    axes[0].plot(history['epoch'], history['train_loss'], style, label=f'{name} treino')
    axes[0].plot(history['epoch'], history['internal_val_loss'], style, marker='o', ms=3, label=f'{name} val. interna')
    axes[1].plot(history['epoch'], history['internal_val_macro_f1'], style, marker='o', ms=3, label=name)
    axes[2].plot(history['epoch'], history['internal_val_covid_recall'], style, marker='o', ms=3, label=name)
for ax, title, ylabel in zip(axes, ['Loss', 'Macro F1 (validação interna)', 'Recall COVID-19 (validação interna)'],
                             ['Loss', 'Macro F1', 'Recall']):
    ax.set_title(title)
    ax.set_xlabel('Época')
    ax.set_ylabel(ylabel)
    ax.legend(fontsize=8)
fig.suptitle('Evolução do treinamento: A × B', fontsize=13)
plt.tight_layout()
save_fig('historico_A_vs_B')
plt.show()

In [ ]:
def wilson_interval(successes: int, n: int, z: float = 1.96):
    """Intervalo de confiança de Wilson para uma proporção (adequado a amostras pequenas)."""
    if n == 0:
        return (np.nan, np.nan)
    p = successes / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denom
    half = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / denom
    return (center - half, center + half)


def paired_bootstrap(pred_a: pd.DataFrame, pred_b: pd.DataFrame, n_boot: int = 2000, seed: int = SEED) -> dict:
    """Bootstrap pareado da diferença B − A no recall de COVID e na macro F1."""
    rng = np.random.default_rng(seed)
    y = pred_a['label'].values
    pa, pb = pred_a['pred'].values, pred_b['pred'].values
    labels = list(range(len(CLASS_NAMES)))
    diffs_recall, diffs_f1 = [], []
    for _ in range(n_boot):
        idx = rng.integers(0, len(y), len(y))
        yi = y[idx]
        positives = yi == COVID_IDX
        if positives.sum() == 0:
            continue
        diffs_recall.append((pb[idx][positives] == COVID_IDX).mean() - (pa[idx][positives] == COVID_IDX).mean())
        diffs_f1.append(f1_score(yi, pb[idx], labels=labels, average='macro', zero_division=0) -
                        f1_score(yi, pa[idx], labels=labels, average='macro', zero_division=0))
    diffs_recall, diffs_f1 = np.array(diffs_recall), np.array(diffs_f1)
    return {'recall_diff_ci': np.percentile(diffs_recall, [2.5, 97.5]).tolist(),
            'macro_f1_diff_ci': np.percentile(diffs_f1, [2.5, 97.5]).tolist(),
            'prob_recall_B_maior': float((diffs_recall > 0).mean())}


pred_A = pd.read_csv(RESULTS_DIR / 'predictions_A.csv')
pred_B = pd.read_csv(RESULTS_DIR / 'predictions_B.csv')
assert (pred_A['path'].values == pred_B['path'].values).all(), 'Validação diferente entre experimentos!'
n_covid_val = int((pred_A['label'] == COVID_IDX).sum())
ci_A = wilson_interval(res_A['metrics']['covid_tp'], n_covid_val)
ci_B = wilson_interval(res_B['metrics']['covid_tp'], n_covid_val)
boot = paired_bootstrap(pred_A, pred_B)
save_json({'wilson_recall_A': ci_A, 'wilson_recall_B': ci_B, **boot}, RESULTS_DIR / 'incerteza_A_B.json')

print(f'Casos COVID-19 na validação real: {n_covid_val} (cada caso = {100 / n_covid_val:.1f} p.p. de recall)')
print(f'Recall COVID A: {res_A["metrics"]["covid_recall"]:.3f}  IC95% Wilson [{ci_A[0]:.3f}, {ci_A[1]:.3f}]')
print(f'Recall COVID B: {res_B["metrics"]["covid_recall"]:.3f}  IC95% Wilson [{ci_B[0]:.3f}, {ci_B[1]:.3f}]')
print(f"Diferença de recall (B − A), IC95% bootstrap: [{100 * boot['recall_diff_ci'][0]:+.1f}, {100 * boot['recall_diff_ci'][1]:+.1f}] p.p.")
print(f"Diferença de macro F1 (B − A), IC95% bootstrap: [{boot['macro_f1_diff_ci'][0]:+.3f}, {boot['macro_f1_diff_ci'][1]:+.3f}]")
print(f"Fração das reamostragens com recall de B > A: {boot['prob_recall_B_maior']:.2f}")

### Respostas às perguntas de comparação

A célula abaixo responde às seis perguntas **a partir dos números desta execução** (nada é escrito manualmente). Critérios usados, definidos antes de ver os resultados:

- variação "relevante" de recall: |Δ| ≥ 1 caso de COVID na validação;
- perda de precisão "importante": queda ≥ 5 p.p. na precisão de COVID ou aumento de falsos positivos;
- acurácia/macro F1 "semelhantes": |Δ| < 2 p.p.;
- evidência "robusta" só se o IC95% bootstrap da diferença de recall excluir zero **e** a macro F1 não piorar — e, mesmo assim, com uma única semente, uma única divisão e sem validação externa, a conclusão permanece preliminar.

In [ ]:
def answer_questions(res_a: dict, res_b: dict, boot: dict, diag: dict, n_pos: int) -> str:
    ma, mb = res_a['metrics'], res_b['metrics']
    one_case = 1 / n_pos
    d_rec = mb['covid_recall'] - ma['covid_recall']
    d_prec = mb['covid_precision'] - ma['covid_precision']
    d_fp = mb['covid_fp'] - ma['covid_fp']
    d_acc = mb['accuracy'] - ma['accuracy']
    d_f1 = mb['macro_f1'] - ma['macro_f1']
    rel = f" ({100 * d_rec / ma['covid_recall']:+.1f}% relativo)" if ma['covid_recall'] > 0 else ' (diferença relativa indefinida: recall de A = 0)'
    out = ['#### Respostas (geradas a partir dos resultados desta execução)\n']
    if QUICK_RUN:
        out.append('> ⚠️ **QUICK_RUN ativo**: as respostas abaixo servem apenas para testar o fluxo.\n')

    # 1
    if abs(d_rec) < one_case - 1e-9:
        verdict = 'Não. O recall de COVID-19 ficou igual'
    elif d_rec > 0:
        verdict = 'Sim, nesta execução. O recall de COVID-19 aumentou'
    else:
        verdict = 'Não. O recall de COVID-19 diminuiu'
    out.append(f"**1. A geração sintética aumentou o recall de COVID-19?** {verdict}: "
               f"{ma['covid_recall']:.3f} → {mb['covid_recall']:.3f} = **{100 * d_rec:+.1f} p.p.**{rel}; "
               f"FN {ma['covid_fn']} → {mb['covid_fn']}.\n")
    # 2
    prec_loss = d_prec <= -0.05 or d_fp > 0
    out.append(f"**2. Houve perda de precisão ou aumento de falsos positivos?** "
               f"{'Sim' if prec_loss else 'Não de forma relevante'}: precisão COVID {ma['covid_precision']:.3f} → "
               f"{mb['covid_precision']:.3f} ({100 * d_prec:+.1f} p.p.); falsos positivos {ma['covid_fp']} → {mb['covid_fp']} ({d_fp:+d}).\n")
    # 3
    acc_trend = 'permaneceu semelhante' if abs(d_acc) < 0.02 else ('melhorou' if d_acc > 0 else 'piorou')
    out.append(f"**3. A acurácia global melhorou, piorou ou permaneceu semelhante?** {acc_trend.capitalize()}: "
               f"{ma['accuracy']:.3f} → {mb['accuracy']:.3f} ({100 * d_acc:+.1f} p.p.).\n")
    # 4
    recalls_a = [ma['per_class'][c]['recall'] for c in CLASS_NAMES]
    recalls_b = [mb['per_class'][c]['recall'] for c in CLASS_NAMES]
    spread_a, spread_b = max(recalls_a) - min(recalls_a), max(recalls_b) - min(recalls_b)
    balanced = d_f1 > 0.02 and spread_b < spread_a
    out.append(f"**4. Os resultados sugerem melhor equilíbrio entre classes?** "
               f"{'Sim, moderadamente' if balanced else 'Não de forma clara'}: macro F1 {ma['macro_f1']:.3f} → {mb['macro_f1']:.3f} "
               f"({100 * d_f1:+.1f} p.p.); amplitude entre o maior e o menor recall por classe {spread_a:.3f} → {spread_b:.3f}.\n")
    # 5
    quality_flags = []
    if diag['razao_diversidade'] < 0.85:
        quality_flags.append(f"diversidade reduzida (razão {diag['razao_diversidade']:.2f})")
    if diag['razao_memorizacao'] < 0.5:
        quality_flags.append(f"proximidade excessiva às imagens de treino (razão {diag['razao_memorizacao']:.2f})")
    quality_flags.append('resolução de 64×64 (perda dos achados finos e possível atalho "imagem borrada ⇒ COVID")')
    out.append(f"**5. A qualidade das imagens sintéticas pode explicar os resultados?** Sim, é um fator plausível: "
               f"{'; '.join(quality_flags)}. Se o recall subiu, parte do ganho pode vir apenas de o modelo prever COVID com mais "
               f"frequência (ver falsos positivos), e não de aprender achados clínicos; se não subiu, as sintéticas provavelmente "
               f"não acrescentaram informação útil além das ~{res_a['train_counts']['COVID']} imagens reais de COVID.\n")
    # 6
    ci_lo, ci_hi = boot['recall_diff_ci']
    robust = (ci_lo > 0 or ci_hi < 0) and d_f1 >= 0
    out.append(f"**6. A evidência é suficiente para uma conclusão robusta?** {'Indício consistente, mas ainda preliminar' if robust else 'Não'}: "
               f"a validação tem apenas {n_pos} casos de COVID (cada caso = {100 * one_case:.1f} p.p.); IC95% bootstrap da diferença "
               f"de recall = [{100 * ci_lo:+.1f}, {100 * ci_hi:+.1f}] p.p. "
               f"{'(exclui zero)' if (ci_lo > 0 or ci_hi < 0) else '(inclui zero)'}. Há uma única semente, uma única divisão, "
               f"nenhum teste externo e nenhuma revisão clínica das sintéticas.\n")
    if d_rec <= 0:
        out.append('**Por que o recall pode não ter melhorado:** baixa diversidade ou colapso de modos da cGAN; artefatos que o '
                   'classificador aprende como "assinatura" das sintéticas e que não existem nas imagens reais; resolução de 64×64 '
                   'insuficiente para representar opacidades; poucas imagens reais de COVID para a GAN aprender; e um classificador '
                   'com backbone congelado, cuja capacidade de aproveitar exemplos adicionais é limitada.\n')
    return '\n'.join(out)


answers_md = answer_questions(res_A, res_B, boot, gan_diagnostics, n_covid_val)
(RESULTS_DIR / 'respostas_comparacao.md').write_text(answers_md, encoding='utf-8')
display(Markdown(answers_md))

## 14. Plano integrado de melhorias

Prioridade definida por **impacto esperado na detecção de COVID-19 / custo**. Cada item deve ser testado isoladamente contra o Experimento A, com a mesma divisão e o mesmo protocolo de avaliação.

### A. Baixo custo — próximo experimento (sem novos dados)

| Prioridade | Medida | Justificativa |
|---|---|---|
| 1 | **Múltiplas sementes e validação cruzada agrupada** (ex.: 5 folds `StratifiedGroupKFold` × 3 sementes) | Com ~24 casos COVID na validação, uma única execução não sustenta conclusões; reportar média ± desvio e IC. |
| 2 | **Class weights** na entropia cruzada ou **amostragem balanceada** (`WeightedRandomSampler`) | Alternativas baratas ao GAN para combater o desbalanceamento; devem ser comparadas com B. |
| 3 | **Análise de limiar e calibração** (curvas PR/ROC por classe, *reliability diagram*, temperature scaling na validação interna) | Permite fixar uma sensibilidade mínima para COVID adequada ao uso (triagem) e expor o custo em falsos positivos. |
| 4 | **Fine-tuning parcial** (descongelar `layer4` com lr menor) | O backbone ImageNet não foi feito para radiografias; ajustar camadas finais costuma ajudar com poucas imagens. |
| 5 | **Augmentation clinicamente plausível** mais rica (pequenas variações de brilho/contraste/gamma, ruído leve, recorte central) — sem espelhamento | Robustez a protocolos de aquisição sem criar anatomia implausível. |
| 6 | **Inspeção de atalhos visuais**: Grad-CAM, oclusão de bordas e textos, treino com pulmões segmentados (as máscaras do próprio dataset), modelo que tenta prever a **fonte** da imagem | Verifica se o classificador olha para os pulmões ou para marcadores/equipamento. |
| 7 | **Melhorar a cGAN** antes de reavaliá-la: maior resolução (128–256), DiffAugment/ADA para poucos dados, varrer `N_SYNTHETIC` (0, 60, 120, 240, 480) **usando apenas validação interna**, e degradar as reais para a mesma resolução como controle | Separa o efeito do conteúdo sintético do efeito "imagem borrada". |
| 8 | Reportar sempre **recall, precisão, F1 por classe, macro F1 e matrizes de confusão** | Evita repetir o erro de olhar só a acurácia global. |

### B. Exigem novos dados, infraestrutura ou validação clínica

| Medida | Justificativa |
|---|---|
| **Divisão por paciente** com identificadores reais (solicitar metadados às fontes ou usar bases com ID de paciente) | Única forma de eliminar o vazamento entre pacientes. |
| **Conjunto de teste independente** congelado, consultado uma única vez | Estimativa imparcial após todas as decisões de desenvolvimento. |
| **Validação externa** em outros hospitais, equipamentos (CR/DR, portátil × fixo) e populações | Mede generalização real e revela atalhos de domínio. |
| **Mais casos reais de COVID-19**, com diversidade de gravidade e fase da doença | Dados reais continuam sendo a fonte mais confiável de informação clínica. |
| **Revisão clínica** por radiologistas: das previsões (sobretudo FN e FP) e das imagens sintéticas (plausibilidade, ausência de achados falsos) | Imagens sintéticas não são evidência clínica; erros precisam de interpretação especializada. |
| **Rótulos de referência** confirmados (RT-PCR + laudo) e análise de concordância entre leitores | Ruído de rótulo limita o desempenho máximo atingível. |
| **Documentação de limitações e riscos** (model card, uso pretendido, populações não avaliadas) e monitoramento pós-implantação | Exigência para qualquer uso assistencial e para avaliação regulatória. |

> Nenhuma das medidas acima, isoladamente, torna o modelo apto ao diagnóstico médico. Mesmo com bons números, uso clínico exigiria validação prospectiva, externa e regulatória.

## 15. Atividade 4.2 — Transfer learning para análise de tráfego urbano

### Cenário

ResNet-50 pré-treinada no ImageNet, fine-tuning com **800 frames rotulados** de **12 câmeras** urbanas; classes *tráfego livre*, *moderado* e *congestionado*; **78% de acurácia** na validação, mas **falhas sistemáticas** à noite, sob chuva e em ângulos de câmera não vistos no treino. Referência indicada: [Transfer learning for traffic analysis (Meegle)](https://www.meegle.com/en_us/topics/transfer-learning/transfer-learning-for-traffic-analysis).

### Problemas técnicos e metodológicos

| # | Problema | Por que acontece | Impacto operacional |
|---|---|---|---|
| 1 | **Poucos dados e baixa diversidade** | 800 frames ÷ 12 câmeras ≈ 67 frames por câmera, provavelmente concentrados em dias claros e horários comerciais. ResNet-50 tem ~25 milhões de parâmetros. | O modelo não aprende a variabilidade real (iluminação, clima, sazonalidade, eventos). Falha justamente nos cenários que mais exigem gestão de tráfego (chuva, noite, acidentes). |
| 2 | **Vazamento e dependência entre frames** | Frames consecutivos da mesma câmera são quase idênticos; uma divisão aleatória por frame coloca "o mesmo instante" no treino e na validação. | Os 78% são **otimistas**: medem memorização da cena, não generalização. Decisões de implantação são tomadas com base em números inflados. |
| 3 | **Mudança de domínio** entre câmeras, ângulos e condições | Cada câmera tem perspectiva, altura, lente, compressão e fundo próprios; noite (faróis, ruído de sensor, IR) e chuva (reflexos, gotas na lente, desfoque) mudam a distribuição de pixels. Features ImageNet foram aprendidas majoritariamente em fotos diurnas. | Degradação silenciosa ao instalar câmeras novas ou ao mudar o clima; semáforos adaptativos ou alertas podem reagir a estados de tráfego errados. |
| 4 | **Ausência de avaliação por condição operacional** | A validação só relata um número agregado. | As falhas noturnas/chuvosas foram descobertas em operação, não na validação. Sem estratificação, não há como definir onde o modelo é confiável. |
| 5 | **Ambiguidade na definição das classes** | "Moderado" é uma faixa contínua entre livre e congestionado; a percepção depende do ângulo, da distância à câmera e do número de faixas. Anotadores diferentes divergem. | Ruído de rótulo limita o teto de desempenho e gera confusões livre↔moderado↔congestionado que não são "erros do modelo", mas da especificação. Indicadores operacionais (ex.: tempo em congestionamento) ficam inconsistentes entre câmeras. |
| 6 | **Uso isolado da acurácia global** | Se a maioria dos frames é "livre", a acurácia é dominada por essa classe. | Pode esconder recall baixo de "congestionado", a classe mais relevante operacionalmente (falso negativo = congestionamento não detectado). |
| 7 | **Ausência de avaliação temporal ou externa** | Validação nos mesmos dias/câmeras do treino. | Não se sabe como o modelo se comporta em outras semanas, estações do ano, obras, mudanças de sinalização ou em câmeras novas. |
| 8 | **Sem adaptação de domínio nem monitoramento pós-implantação** | Ambiente urbano muda continuamente (deriva de dados: câmeras reposicionadas, lentes sujas, troca de equipamento). | O desempenho degrada sem que ninguém perceba; falhas só aparecem quando já causaram decisões erradas. |

### Medidas corretivas

1. **Separação por câmera / leave-one-camera-out (LOCO):** treinar com 11 câmeras e avaliar na 12ª, repetindo para todas. Mede diretamente a generalização para câmeras e ângulos novos. Complementar com **divisão temporal** (treinar em semanas anteriores, validar em semanas posteriores) e blocos de tempo para evitar frames vizinhos em conjuntos diferentes.
2. **Dados reais de noite, chuva e novos ângulos:** coleta direcionada (amostragem estratificada por câmera × período × clima), priorizando as condições em que o modelo falha; aprendizado ativo para escolher quais frames rotular.
3. **Definições consistentes de rótulo:** protocolo de anotação com critérios objetivos (ex.: ocupação da via, velocidade média, densidade por faixa, fila em uma região de interesse fixa por câmera), exemplos de referência por câmera, dupla anotação e medida de concordância (kappa de Cohen). Considerar regressão de densidade/ocupação e discretizar depois.
4. **Métricas adequadas:** precisão, recall e F1 por classe, **macro F1** e **matriz de confusão**; dar destaque ao recall de "congestionado"; considerar custo assimétrico entre erros adjacentes (livre↔moderado) e extremos (livre↔congestionado).
5. **Avaliação estratificada por câmera, período do dia e condição climática:** reportar métricas por fatia (dia/noite, seco/chuva, cada câmera) e definir um desempenho mínimo aceitável por fatia, não apenas na média.
6. **Data augmentation e adaptação de domínio quando justificadas:** variações de brilho/gamma, ruído de sensor, desfoque, simulação de gotas e reflexos, perspectiva leve; ajuste de BatchNorm ao domínio-alvo, adaptação não supervisionada (ex.: DANN, pseudo-rótulos) ou tradução de domínio dia→noite — sempre validadas em dados **reais** de noite/chuva, nunca apenas em dados aumentados ou sintéticos.
7. **Monitoramento pós-implantação:** detectar deriva de dados (estatísticas de imagem, distribuição de confiança, proporção de classes por câmera), auditoria periódica com rótulos humanos, alertas de baixa confiança, comparação com sensores independentes (laços indutivos, dados de GPS/velocidade) e re-treinamento planejado com versionamento.

### Por que 78% de acurácia na validação não garante desempenho confiável

- O número é uma **média** sobre a distribuição da validação. Se a validação tem as mesmas câmeras, ângulos, dias e condições do treino (e frames quase idênticos), ela mede a capacidade de **reconhecer cenas já vistas**, não de operar em condições novas.
- A acurácia global **não mostra onde** os 22% de erro estão. As falhas sistemáticas observadas (noite, chuva, ângulos novos) indicam que os erros estão **concentrados** em fatias específicas — que podem ser exatamente as mais críticas operacionalmente.
- Sob **mudança de domínio**, a relação entre pixels e classes muda; o desempenho em um domínio não é evidência sobre outro. Em uma noite chuvosa numa câmera nova, a acurácia real pode ser muito inferior a 78%.
- A acurácia também depende da **proporção de classes**, da **qualidade dos rótulos** e de possível **vazamento entre frames**, podendo superestimar o desempenho.

Conclusão: confiabilidade operacional exige avaliação por câmera (LOCO), por condição e ao longo do tempo, com métricas por classe e monitoramento contínuo após a implantação.

## 16. Conclusão e limitações

A célula abaixo resume os resultados **efetivamente obtidos** nesta execução e os tempos de cada etapa. A interpretação deve sempre considerar as limitações listadas em seguida.

In [ ]:
summary_md = [f'### Resumo desta execução (semente {SEED}{", QUICK_RUN" if QUICK_RUN else ""})\n']
summary_md.append(f'- Amostra: {len(sample_df)} imagens reais (840/240/120); treino de ajuste {len(train_fit_df)}, '
                  f'validação interna {len(internal_val_df)}, validação real {len(val_df)} '
                  f'({n_covid_val} casos de COVID-19).')
summary_md.append(f'- cGAN: {GAN_EPOCHS} épocas a {GAN_IMAGE_SIZE}×{GAN_IMAGE_SIZE}; {N_SYNTHETIC} imagens sintéticas de COVID-19; '
                  f"razão de diversidade {gan_diagnostics['razao_diversidade']:.2f}, razão de memorização {gan_diagnostics['razao_memorizacao']:.2f}.")
for name, res in [('A', res_A), ('B', res_B)]:
    m = res['metrics']
    summary_md.append(f"- Experimento {name}: acurácia {m['accuracy']:.3f} | macro F1 {m['macro_f1']:.3f} | "
                      f"recall COVID {m['covid_recall']:.3f} | precisão COVID {m['covid_precision']:.3f} | "
                      f"F1 COVID {m['covid_f1']:.3f} | checkpoint da época {res['best_epoch']}.")
d_rec = res_B['metrics']['covid_recall'] - res_A['metrics']['covid_recall']
summary_md.append(f"- Diferença absoluta de recall de COVID-19 (B − A): **{100 * d_rec:+.1f} p.p.** "
                  f"(IC95% bootstrap [{100 * boot['recall_diff_ci'][0]:+.1f}, {100 * boot['recall_diff_ci'][1]:+.1f}] p.p.).")
summary_md.append('\n**Tempos de execução:**\n')
summary_md += [f'- {name}: {seconds / 60:.1f} min' for name, seconds in EXEC_TIMES.items()]
summary_text = '\n'.join(summary_md)
(RESULTS_DIR / 'resumo_execucao.md').write_text(summary_text, encoding='utf-8')
save_json({'exec_times_seconds': EXEC_TIMES, 'gan_diagnostics': gan_diagnostics}, RESULTS_DIR / 'execucao.json')
display(Markdown(summary_text))
print(f'Artefatos salvos em: {RESULTS_DIR}')

### Conclusão

- O cenário original combina desbalanceamento, avaliação apenas por acurácia, treinamento do zero com poucos dados e ausência de controles de vazamento — o que explica, por si só, a percepção de que o modelo "ignora" COVID-19. Neste notebook, esses fatores foram tratados com seleção reprodutível, divisão estratificada e agrupada, transfer learning, augmentation moderada, seleção de checkpoint em validação interna e métricas por classe.
- O efeito das imagens sintéticas deve ser julgado **pela tabela da Seção 13 e pelas respostas geradas automaticamente**, considerando conjuntamente recall, precisão, falsos positivos, macro F1 e a incerteza estatística. Um aumento de recall acompanhado de mais falsos positivos ou de queda da macro F1 não caracteriza melhora; uma diferença cujo intervalo de confiança inclui zero não caracteriza efeito.
- Independentemente do resultado, a cGAN em 64×64 treinada com menos de 100 radiografias reais de COVID-19 é uma abordagem **exploratória**. Suas imagens não foram validadas clinicamente e não constituem evidência médica.

### Limitações

1. **Sem ID de paciente:** o agrupamento por hash perceptual reduz, mas não elimina, o risco de vazamento entre pacientes.
2. **Validação pequena:** ~24 casos de COVID-19; cada erro altera o recall em ~4 p.p.
3. **Uma única semente e uma única divisão:** variações entre execuções não foram quantificadas (ver plano, item A1).
4. **Sem teste independente nem validação externa:** os resultados valem apenas para esta amostra deste dataset.
5. **Atalhos de domínio:** as classes vêm de fontes distintas (ver tabela de fontes); o classificador pode explorar diferenças de origem.
6. **cGAN limitada:** resolução 64×64, poucos exemplos de COVID-19, sem métrica de qualidade validada para radiografias (ex.: FID em features médicas) e sem revisão por radiologista; diferença de resolução entre reais e sintéticas pode criar atalhos.
7. **Baseline diferente do cenário:** transfer learning com backbone congelado não reproduz a ResNet-18 treinada do zero; os números não são comparáveis aos 93%/61% do enunciado.
8. **Amostra com proporções artificiais** (70/20/10), diferentes do dataset original e de qualquer prevalência clínica real.

> **Este modelo não está pronto para diagnóstico médico** e não deve ser usado para apoiar decisões clínicas.